<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.es/cap08/cap08.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Práctica con Ejercicios de Programación**

La presente lista de ejercicios de programación (EP) consolida las formulaciones teóricas presentadas a lo largo del Capítulo 8 — Correspondencia de Características, Detección de Objetos y Segmentación Clásica — mediante una ruta práctica aplicada. Así como en el capítulo anterior, los EPs aíslan las **magnitudes intermedias** de cada técnica — la distancia entre descriptores binarios, los términos de una imagen integral, el conteo de *inliers* de un modelo candidato, la superposición entre cajas delimitadoras y la etiqueta de cada componente conectado — permitiendo validar manualmente cada etapa del razonamiento sin depender de OpenCV ni de imágenes externas.

El encadenamiento de los ejercicios reproduce el flujo conceptual del capítulo: se inicia con la **distancia de Hamming**, corazón de la correspondencia de descriptores binarios como el ORB; se avanza hacia el conteo de ***inliers*** que sustenta el **RANSAC** en la estimación robusta de una homografía; se continúa con la **imagen integral**, el truco computacional que hace viable al ***Haar Cascade*** en tiempo real; se profundiza en **IoU y Supresión de No-Máximos**, el post-procesamiento común a prácticamente todo detector de objetos; y se concluye con el **etiquetado de componentes conectados**, el enfoque clásico — y sus limitaciones — para segmentar instancias individuales en una máscara binaria.

### 🎯 Objetivo de este Cuaderno

El cuaderno permite desarrollar, validar, organizar y probar soluciones de **Ejercicios de Programación (EPs)** en entornos interactivos, como Colab, con los mismos casos de prueba de Moodle, copiándolos allí solo al momento de registrar la nota oficial.

### *Download*

Descargue `morph.py` y `testsuite.py` ejecutando la celda siguiente:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Ejecutando las pruebas
Para evaluar las pruebas, ejecute `TestSuite("EP08_01.extensión").run()` en una nueva celda, reemplazando la extensión por la del lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema descarga los casos de prueba de GitHub, ejecuta el programa y calcula la nota automáticamente.

Para probar código Python directamente, sin guardar un archivo, use `run_code(codigo)` pasando el código como *string* en una variable `codigo`:

```python
codigo = """
# ... su código aquí ...
"""
TestSuite("EP08_01").run_code(codigo)
```

### 🛠️ Resumen de los Métodos de `morph.py` (Cap. 8)

La biblioteca `morph.py` proporciona funciones para el análisis de componentes conexos, extracción de contornos, métricas geométricas y anotaciones:

1. **Componentes y Contornos (`connectedComponents`, `findContours`)**
Etiquetan regiones conexas y extraen los contornos de imágenes binarias.
2. **Propiedades del Contorno (`contourArea`, `arcLength`, `convexHull`, `approxPolyDP`, `fitLine`)**
Calculan área, perímetro, envolvente convexa, aproximación poligonal y ajuste de recta para un contorno.
3. **Geometría Envolvente (`boundingRect`, `minAreaRect`, `boxPoints`, `minEnclosingCircle`, `fitEllipse`)**
Determinan rectángulos envolventes (alineados u orientados), elipses y el círculo delimitador más pequeño.
4. **Extracción y Persistencia de Medidas (`measure`, `saveMeasures`)**
Extraen descriptores geométricos de los objetos (área, circularidad, solidez, centroide) y exportan los datos a CSV, texto o formato YOLO.
5. **Evaluación y Visualización (`IoU`, `verifyBoundBox`, `showBoundBox`)**
Calculan la intersección sobre unión (*Intersection over Union*), validan cajas delimitadoras con patrones de referencia y dibujan cajas delimitadoras anotadas sobre la imagen.

### EP08_01 🟢 Distancia de Hamming y Correspondencia de Descriptores Binarios

ORB, utilizado en el Proyecto Práctico 1 de este capítulo, describe la vecindad de cada punto de interés como una secuencia de bits — y, por ello, la comparación entre dos descriptores no utiliza la distancia euclidiana del k-NN del Capítulo 7, sino la **distancia de Hamming**: el número de posiciones en que los bits difieren. Antes de llamar a `cv2.BFMatcher(cv2.NORM_HAMMING)`, se te encargó implementar manualmente esta correspondencia (*matching*) por fuerza bruta — la misma etapa que, ejecutada internamente por OpenCV, precede a la estimación robusta de la homografía mediante RANSAC.

#### 📋 Directrices de Implementación

1. **Cantidades:** Leer los enteros $N$ y $M$ — número de descriptores extraídos de la imagen A y de la imagen B, respectivamente.
2. **Descriptores de A:** Leer $N$ líneas, cada una conteniendo un descriptor binario (una *cadena* de caracteres `0` y `1`, todos de la misma longitud).
3. **Descriptores de B:** Leer $M$ líneas, en el mismo formato.
4. **Umbral:** Leer el entero $\tau$ — distancia de Hamming máxima aceptable para considerar una correspondencia válida.
5. **Distancia de Hamming:** Para dos descriptores binarios $a$ y $b$ de igual longitud,
$$
d_H(a, b) = \sum_{k} \mathbb{1}[a_k \neq b_k],
$$
   es decir, el conteo de posiciones en que los bits difieren.
6. **Correspondencia por vecino más cercano:** Para cada descriptor $a_i$ de A ($i$ en el orden de lectura, comenzando en $0$), calcula su distancia de Hamming a **todos** los descriptores de B y encuentra el de menor distancia. En caso de empate entre dos o más descriptores de B con la misma distancia mínima, elige el de **menor índice**.
7. **Filtrado por umbral:** Si la menor distancia encontrada es $\le \tau$, la correspondencia es válida; de lo contrario, $a_i$ no posee correspondencia.
8. **Salida:** Para cada $i$ de $0$ a $N-1$, en el orden de lectura, imprimir una línea: `i j d` si hay correspondencia válida (donde $j$ es el índice del descriptor de B elegido y $d$ su distancia), o `i -1` en caso contrario. Al final, imprimir `Total correspondencias válidas: X`.

#### 📌 Restricciones Computacionales

* **Misma longitud:** todos los descriptores (de A y de B) tienen exactamente el mismo número de bits.
* **Fuerza bruta:** compara cada descriptor de A con **todos** los de B — no se necesita ningún tipo de indexación ni estructura de aceleración.
* **Desempate por menor índice en B**, y **nunca** por orden de lectura de A (que ya es natural, pues cada $a_i$ se trata de forma independiente).

#### 🧠 Fundamentación Teórica

| Elemento | Papel en la correspondencia ORB |
|---|---|
| Descriptor binario (BRIEF) | Cada bit es el resultado de una comparación de intensidad entre dos píxeles de la vecindad |
| Distancia de Hamming | Métrica de disimilitud entre *cadenas* binarias; mucho más rápida de calcular que la distancia euclidiana (operación XOR + conteo de bits) |
| Vecino más cercano | Criterio de correspondencia: cada punto de A se empareja con el punto de B cuyo descriptor sea más similar |
| Umbral $\tau$ | Filtra correspondencias poco confiables incluso antes del RANSAC — pero, como se discutió en el capítulo, algunas correspondencias incorrectas aún pasan, exigiendo la robustez del RANSAC |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $N$ y $M$.
* Siguientes $N$ líneas: un descriptor binario por línea (*cadena* de `0`s y `1`s).
* Siguientes $M$ líneas: un descriptor binario por línea, en el mismo formato.
* Última línea: Entero $\tau$.

**Salida:**

* $N$ líneas, una por descriptor de A, en el formato `i j d` o `i -1`.
* Última línea: `Total correspondencias válidas: X`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3 3<br>10101010<br>11110000<br>00001111<br>10101011<br>00001110<br>11111111<br>2 | 0 0 1<br>1 -1<br>2 1 1<br>Total correspondencias válidas: 2 | El descriptor `11110000` no encuentra correspondencia: su vecino más cercano está a distancia 4, por encima del umbral $\tau=2$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0801" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0801 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0801 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0801 button:hover { background: #e8dfcf; }
  .sim-ep0801_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0801_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0801_bit { width: 36px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-family: monospace; font-weight: 700; font-size: 14px; user-select: none; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP08_01: Distancia de Hamming entre Descriptores Binarios</span>
  <span class="sim-ep0801_pill">Descriptores de 8 Bits</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel Informativo de Instruções -->
  <div class="sim-ep0801_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center;">
      Haz clic en cualquier bit del <b>Descriptor B</b> para invertirlo y observa cómo cambia la distancia de Hamming en tiempo real.
    </div>
  </div>

  <!-- Grades dos Descritores -->
  <div class="sim-ep0801_panel" style="margin-bottom:14px; display:flex; flex-direction:column; gap:12px; align-items:center;">
    <div>
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:6px; text-align:center; letter-spacing:0.04em;">
        Descriptor A (Fijo)
      </div>
      <div id="sim-ep0801_a" style="display:grid; grid-template-columns:repeat(8, 36px); gap:4px;"></div>
    </div>

    <div>
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:6px; text-align:center; letter-spacing:0.04em;">
        Descriptor B (Clic para Invertir)
      </div>
      <div id="sim-ep0801_b" style="display:grid; grid-template-columns:repeat(8, 36px); gap:4px;"></div>
    </div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0801_debug" class="sim-ep0801_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep01(root){
    if (!root || root.dataset.sim08Ep01Init) return;
    root.dataset.sim08Ep01Init = "1";

    var A = [1, 0, 1, 0, 1, 0, 1, 0];
    var B = [1, 0, 1, 0, 1, 0, 1, 1];

    var aEl = root.querySelector('#sim-ep0801_a');
    var bEl = root.querySelector('#sim-ep0801_b');
    var dbg = root.querySelector('#sim-ep0801_debug');

    function estiloBit(destacado, interativo){
      var base = destacado 
        ? 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;' 
        : 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;';
      var cursor = interativo ? ' cursor:pointer;' : ' cursor:default;';
      return base + cursor;
    }

    function render(){
      aEl.innerHTML = ''; 
      bEl.innerHTML = '';
      var dist = 0;

      for (var k = 0; k < 8; k++){
        var diff = A[k] !== B[k];
        if (diff) dist++;

        var da = document.createElement('div');
        da.className = 'sim-ep0801_bit';
        da.style.cssText = estiloBit(diff, false);
        da.textContent = A[k];
        aEl.appendChild(da);

        var db = document.createElement('div');
        db.className = 'sim-ep0801_bit';
        db.style.cssText = estiloBit(diff, true);
        db.textContent = B[k];
        
        (function(idx){
          db.addEventListener('click', function(){
            B[idx] = 1 - B[idx];
            render();
          });
        })(k);

        bEl.appendChild(db);
      }

      if (dist === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#26241d';
      }

      dbg.textContent = 'A = ' + A.join('') + '   B = ' + B.join('') + '   →   Distância de Hamming = ' + dist;
    }

    render();
  }

  function tryInitSim08Ep01(){
    var root = document.getElementById('sim-ep0801');
    if (root) initSim08Ep01(root); else setTimeout(tryInitSim08Ep01, 200);
  }
  tryInitSim08Ep01();
})();
</script>
""")

**Figura 8.1:** Simulador EP08_01: Distancia de Hamming entre Dos Descriptores Binarios


In [ ]:
%%writefile EP08_01.py
# Código Python

In [ ]:
TestSuite("EP08_01.py").run()

### EP08_02 🟢 Homografía y RANSAC: La Votación por *Inliers*

El RANSAC, presentado en la sección "Modelado Matemático: Homografía y RANSAC", repite un ciclo de tres pasos — sortear una muestra mínima, estimar un modelo candidato y contar cuántas correspondencias son consistentes con él (los ***inliers***) — manteniendo al final el modelo más votado. La etapa de estimación del modelo a partir de 4 puntos (paso 2) involucra álgebra lineal que queda fuera del alcance de este EP; aquí, recibes directamente un conjunto de homografías **ya candidatas** — como si cada una hubiera sido estimada a partir de una muestra aleatoria diferente — y tienes la tarea de reproducir exactamente el paso decisivo del algoritmo: **aplicar cada modelo a todas las correspondencias y contar sus *inliers***, eligiendo al ganador.

#### 📋 Directrices de Implementación

1. **Correspondencias:** Leer el entero $N$ y, a continuación, $N$ líneas con cuatro reales cada una, $x\ y\ x'\ y'$ — un punto de la imagen A y su correspondiente (posiblemente incorrecto) en la imagen B, exactamente como lo produce la etapa de *matching* del EP08_01.
2. **Modelos candidatos:** Leer el entero $K$ (número de homografías candidatas) y el real $\varepsilon$ (umbral de error de reproyección). Luego, leer $K$ líneas, cada una con nueve reales $h_{11}\ h_{12}\ h_{13}\ h_{21}\ h_{22}\ h_{23}\ h_{31}\ h_{32}\ h_{33}$ — los elementos de la matriz $H$ candidata, en orden de lectura por fila (*row-major*).
3. **Reproyección:** Para cada correspondencia $(x,y,x',y')$ y cada modelo candidato $H_k$, calcular el punto proyectado
$$
\begin{bmatrix} \hat x \\ \hat y \\ \hat w \end{bmatrix} = H_k \begin{bmatrix} x \\ y \\ 1 \end{bmatrix},
\qquad
(\hat x / \hat w,\ \hat y / \hat w)\ \text{es el punto proyectado.}
$$
4. **Error de reproyección:** $e = \sqrt{(\hat x/\hat w - x')^2 + (\hat y /\hat w - y')^2}$.
5. **Conteo de *inliers*:** Una correspondencia es un *inlier* del modelo $H_k$ si $e \le \varepsilon$.
6. **Selección del mejor modelo:** El modelo ganador es el que tiene mayor número de *inliers*; en caso de empate, elige el de **menor índice** $k$ (el primero encontrado durante el ciclo iterativo del RANSAC).
7. **Salida:** Para cada modelo $k$ de $0$ a $K-1$, en el orden de lectura, imprime `Modelo k: I inliers`. Al final, imprime `Mejor modelo: k_best con I_best inliers`.

#### 📌 Restricciones Computacionales

* **Comparación inclusiva:** un error de reproyección **exactamente igual** a $\varepsilon$ cuenta como *inlier* ($e \le \varepsilon$).
* **Sin estimación de $H$:** las matrices ya se proporcionan listas — no es necesario (ni esperado) resolver ningún sistema lineal.
* **Empate resuelto por el menor índice**, reflejando el comportamiento natural de un algoritmo iterativo que recorre los modelos en orden y solo reemplaza al mejor encontrado hasta entonces cuando un nuevo modelo lo **supera estrictamente**.

#### 🧠 Fundamentación Teórica

| Elemento | Papel en el RANSAC |
|---|---|
| Muestra mínima (4 pares) | Suficiente para determinar los 8 grados de libertad de una homografía |
| Modelo candidato $H_k$ | Estimado a partir de una muestra mínima; puede ser bueno o malo, dependiendo de si la muestra contenía *outliers* |
| Error de reproyección | Mide qué tan bien el modelo "predice" cada correspondencia observada |
| *Inlier* vs. *outlier* | Correspondencias consistentes con el modelo ganador (*inliers*) vs. las demás, típicamente correspondencias incorrectas del *matching* |
| Refinamiento final | En la práctica, tras elegir el mejor modelo, el RANSAC lo recalcula usando **solo** sus *inliers* — paso no exigido en este EP |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $N$.
* Siguientes $N$ líneas: cuatro reales $x\ y\ x'\ y'$.
* Siguiente línea: Entero $K$ y real $\varepsilon$.
* Siguientes $K$ líneas: nueve reales (elementos de $H_k$, *row-major*).

**Salida:**

* $K$ líneas en el formato `Modelo k: I inliers`.
* Última línea: `Mejor modelo: k_best con I_best inliers`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 5<br>0 0 0 0<br>1 1 2 2<br>2 0 4 0<br>0 2 0 4<br>5 5 1 1<br>2 0.5<br>2 0 0 0 2 0 0 0 1<br>1 0 0 0 1 0 0 0 1 | Modelo 0: 4 inliers<br>Modelo 1: 1 inliers<br>Mejor modelo: 0 con 4 inliers | El Modelo 0 (escala ×2) explica correctamente 4 de las 5 correspondencias; la 5ª, $(5,5)\to(1,1)$, es un *outlier* que ninguno de los dos modelos explica bien. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0802" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0802 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0802 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0802 button:hover { background: #e8dfcf; }
  #sim-ep0802 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0802_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0802_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP08_02: RANSAC &mdash; Conteo de Inliers</span>
  <span class="sim-ep0802_pill">Modelo: Escala &times;2</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0808_panel sim-ep0802_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral (&epsilon;): <span id="sim-ep0802_vl" style="font-family:monospace; color:#26241d;">0.50</span>
      </label>
    </div>
    
    <input id="sim-ep0802_sl" type="range" min="0" max="13" step="0.25" value="0.5">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      El modelo candidato mapea (x,y) &rarr; (2x,2y). Ajuste el umbral &epsilon; y vea qué correspondencias se vuelven inliers o outliers.
    </div>
  </div>

  <!-- Cards de Pontos / Correspondências -->
  <div id="sim-ep0802_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:8px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0802_debug" class="sim-ep0802_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep02(root){
    if (!root || root.dataset.sim08Ep02Init) return;
    root.dataset.sim08Ep02Init = "1";

    var pontos = [
      {x:0, y:0, xp:0, yp:0},
      {x:1, y:1, xp:2, yp:2},
      {x:2, y:0, xp:4, yp:0},
      {x:0, y:2, xp:0, yp:4},
      {x:5, y:5, xp:1, yp:1}
    ];

    var slEl  = root.querySelector('#sim-ep0802_sl');
    var vlEl  = root.querySelector('#sim-ep0802_vl');
    var cards = root.querySelector('#sim-ep0802_cards');
    var dbg   = root.querySelector('#sim-ep0802_debug');

    function render(){
      var eps = parseFloat(slEl.value);
      vlEl.textContent = eps.toFixed(2);
      cards.innerHTML = '';
      var inliers = 0;

      pontos.forEach(function(p){
        var px = 2 * p.x, py = 2 * p.y;
        var erro = Math.sqrt((px - p.xp) * (px - p.xp) + (py - p.yp) * (py - p.yp));
        var dentro = erro <= eps;
        if (dentro) inliers++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
            : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">(' + p.x + ',' + p.y + ') &rarr; (' + p.xp + ',' + p.yp + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">erro = ' + erro.toFixed(2) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'INLIER' : 'outlier') + '</div>';

        cards.appendChild(div);
      });

      if (inliers > 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = '\u03B5 = ' + eps.toFixed(2) + '  |  inliers = ' + inliers + ' de ' + pontos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim08Ep02(){
    var root = document.getElementById('sim-ep0802');
    if (root) initSim08Ep02(root); else setTimeout(tryInitSim08Ep02, 200);
  }
  tryInitSim08Ep02();
})();
</script>
""")

**Figura 8.2:** Simulador EP08_02: RANSAC — Votación por Inliers entre Modelos Candidatos


In [ ]:
%%writefile EP08_02.py
# Código Python

In [ ]:
TestSuite("EP08_02.py").run()

### EP08_03 🟢 Imagen Integral: Sumas Rectangulares en Tiempo Constante

Imagina una cámara de seguridad procesando 30 fotogramas por segundo, y para cada fotograma el sistema necesita recorrer la imagen en decenas de posiciones y escalas diferentes, probando en cada una un conjunto de características rectangulares para decidir "¿hay una cara aquí?". Si calcular la suma de intensidades de cada rectángulo requiriese sumar píxel a píxel, el sistema no tendría la menor oportunidad de evaluar en tiempo real — el cuello de botella estaría justamente en la parte más repetida del algoritmo. Es exactamente ese cuello de botella el que la imagen integral elimina.

El Haar Cascade evalúa miles de características rectangulares por ventana, en múltiples posiciones y escalas — algo inviable en tiempo real si cada rectángulo requiriese sumar sus píxeles uno a uno. La **imagen integral**, definida en la sección sobre Haar Cascade, resuelve este problema: una vez precomputada, la suma de intensidades de **cualquier** región rectangular se obtiene con solo cuatro consultas y tres operaciones aritméticas, independientemente del tamaño del rectángulo.

Se te ha encargado implementar esta estructura desde cero: primero, calcular la imagen integral a partir de la imagen original; luego, responder a consultas rectangulares arbitrarias.

#### 📋 Directrices de Implementación

1. **Entrada:** Leer las dimensiones $H \times W$ de la imagen y sus $H \times W$ valores enteros de intensidad.
2. **Imagen integral:** Calcular, para cada posición $(i,j)$ (indexación desde $0$, `[fila][columna]`),
$$
II(i,j) = \sum_{i' \le i,\ j' \le j} I(i', j'),
$$
   es decir, la suma de todos los píxeles arriba y a la izquierda de $(i,j)$, incluyendo la propia posición.
3. **Consultas:** Leer el entero $Q$ y, a continuación, $Q$ líneas, cada una con cuatro enteros $x_1\ y_1\ x_2\ y_2$ — las esquinas superior-izquierda e inferior-derecha de un rectángulo, **ambas inclusivas**, con $0 \le x_1 \le x_2 < W$ y $0 \le y_1 \le y_2 < H$.
4. **Suma rectangular en O(1):** Para cada consulta, calcular la suma de intensidades dentro del rectángulo usando exclusivamente valores ya presentes en $II$ (sin recorrer los píxeles originales):
$$
S(x_1,y_1,x_2,y_2) = II(y_2,x_2) - II(y_2, x_1{-}1) - II(y_1{-}1, x_2) + II(y_1{-}1, x_1{-}1),
$$
   tratando cualquier término con índice de fila o columna igual a $-1$ como $0$.
5. **Salida:** Primero, imprimir la imagen integral completa — $H$ líneas con $W$ enteros cada una. Luego, para cada consulta, imprimir un único entero: la suma de la región correspondiente.

#### 📌 Restricciones Computacionales

* **No recalcular por fuerza bruta:** la respuesta a cada consulta debe usar la fórmula de cuatro términos sobre $II$, no una suma directa de los píxeles del rectángulo (aunque el resultado numérico sea el mismo, el objetivo del ejercicio es justamente esa técnica).
* **Rectángulos con coordenadas inclusivas:** $(x_1,y_1)$ y $(x_2,y_2)$ pertenecen a la región sumada.
* **Tratamiento de borde:** al consultar $II$ con índice $-1$ (cuando $x_1=0$ o $y_1=0$), utilizar el valor $0$.

#### 🧠 Fundamentación Teórica

| Elemento | Papel en el Haar Cascade |
|---|---|
| Imagen integral $II$ | Precomputada una única vez por imagen, en tiempo $O(HW)$ |
| Consulta en O(1) | Cada característica Haar (diferencia entre sumas de regiones rectangulares) se evalúa con pocas operaciones, independientemente del área del rectángulo |
| Escalabilidad | Es esa constancia la que hace viable evaluar miles de características, en múltiples posiciones y escalas, en tiempo real |
| Principio de inclusión-exclusión | Los cuatro términos de la fórmula suman la región deseada y restan exactamente las áreas contadas en exceso |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $H$ y $W$.
* Siguientes $H$ líneas: $W$ enteros cada una (imagen original).
* Siguiente línea: Entero $Q$.
* Siguientes $Q$ líneas: cuatro enteros $x_1\ y_1\ x_2\ y_2$.

**Salida:**

* $H$ líneas con $W$ enteros cada una (la imagen integral).
* $Q$ líneas, una por consulta, con la suma de la región correspondiente.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>1<br>0 0 2 2 | 1 3 6<br>5 12 21<br>12 27 45<br>45 | La consulta cubre la imagen completa; la suma coincide con $II(2,2)$ y con la suma de todos los 9 valores. |
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>2<br>1 1 2 2<br>0 0 1 1 | 1 3 6<br>5 12 21<br>12 27 45<br>28<br>12 | La primera consulta usa los cuatro términos de la fórmula; la segunda coincide directamente con $II(1,1)$, pues comienza en el origen. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0803" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0803 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0803 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0803 button:hover { background: #e8dfcf; }
  #sim-ep0803 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0803_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0803_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP08_03: Soma Rectangular con Imagen Integral</span>
  <span id="sim-ep0803_badge" class="sim-ep0803_pill">Interno</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição e Botões de Preset -->
  <div class="sim-ep0803_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Elija un rectángulo (x<sub>1</sub>, y<sub>1</sub>) &ndash; (x<sub>2</sub>, y<sub>2</sub>). La imagen integral II incluye borde virtual (&minus;1) con ceros para validación sin excepciones.
    </div>

    <div style="display:flex; gap:6px; justify-content:center; flex-wrap:wrap;">
      <button data-preset="0,0,3,3">Desde el Origen</button>
      <button data-preset="0,1,2,3">Borde Izquierdo</button>
      <button data-preset="1,0,3,2">Borde Superior</button>
      <button data-preset="1,1,2,2">Totalmente Interno</button>
      <button data-preset="2,2,2,2">Píxel Único</button>
    </div>
  </div>

  <!-- Sliders de Seleção das Coordenadas -->
  <div class="sim-ep0803_panel" style="margin-bottom:14px;">
    <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:12px;">
      
      <div>
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:4px;">
          Esquina Superior-Izquierda (x<sub>1</sub>, y<sub>1</sub>) = <span id="sim-ep0803_v_tl" style="font-family:monospace; color:#26241d;">(1,1)</span>
        </div>
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600;">x<sub>1</sub></div>
        <input id="sim-ep0803_x1" type="range" min="0" max="3" step="1" value="1">
        
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600; margin-top:4px;">y<sub>1</sub></div>
        <input id="sim-ep0803_y1" type="range" min="0" max="3" step="1" value="1">
      </div>

      <div>
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:4px;">
          Esquina Inferior-Derecha (x<sub>2</sub>, y<sub>2</sub>) = <span id="sim-ep0803_v_br" style="font-family:monospace; color:#26241d;">(2,2)</span>
        </div>
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600;">x<sub>2</sub></div>
        <input id="sim-ep0803_x2" type="range" min="0" max="3" step="1" value="2">
        
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600; margin-top:4px;">y<sub>2</sub></div>
        <input id="sim-ep0803_y2" type="range" min="0" max="3" step="1" value="2">
      </div>

    </div>
  </div>

  <!-- Grades das Matrizes -->
  <div style="display:flex; gap:20px; justify-content:center; flex-wrap:wrap; margin-bottom:14px;">
    <div class="sim-ep0803_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:8px; letter-spacing:0.04em;">
        Imagen Original I (4&times;4)
      </div>
      <div id="sim-ep0803_gridI" style="display:grid; justify-content:center;"></div>
    </div>

    <div class="sim-ep0803_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:8px; letter-spacing:0.04em;">
        Imagen Integral II (Con Borde Virtual &minus;1)
      </div>
      <div id="sim-ep0803_gridII" style="display:grid; justify-content:center;"></div>
    </div>
  </div>

  <!-- Legenda das Operações -->
  <div style="display:flex; gap:12px; justify-content:center; flex-wrap:wrap; margin-bottom:14px; font-size:10px; font-weight:700; color:#5e5a4a;">
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#2980b9; border-radius:2px; display:inline-block;"></span> + II(y<sub>2</sub>, x<sub>2</sub>)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#d35400; border-radius:2px; display:inline-block;"></span> &minus; II(y<sub>2</sub>, x<sub>1</sub>&minus;1)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#d35400; border-radius:2px; display:inline-block;"></span> &minus; II(y<sub>1</sub>&minus;1, x<sub>2</sub>)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#27ae60; border-radius:2px; display:inline-block;"></span> + II(y<sub>1</sub>&minus;1, x<sub>1</sub>&minus;1)</span>
  </div>

  <!-- Painéis Informativos / Resultados -->
  <div id="sim-ep0803_formula" class="sim-ep0803_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center; margin-bottom:8px;"></div>
  <div id="sim-ep0803_verify" class="sim-ep0803_panel" style="font-family:monospace; font-size:11px; color:#04342C; background:#eafaf1; border-color:#a3e4d7; text-align:center;"></div>

</div>
</div>

<script>
(function(){
  function initSim08Ep03(root){
    if (!root || root.dataset.sim08Ep03Init) return;
    root.dataset.sim08Ep03Init = "1";

    var I = [
      [2, 1, 3, 4],
      [5, 6, 1, 2],
      [3, 2, 4, 1],
      [1, 3, 2, 5]
    ];
    var N = 4;
    var II = [];
    for (var i = 0; i < N; i++){ II.push([0, 0, 0, 0]); }
    
    for (var i = 0; i < N; i++){
      for (var j = 0; j < N; j++){
        II[i][j] = I[i][j] +
          (i > 0 ? II[i - 1][j] : 0) + 
          (j > 0 ? II[i][j - 1] : 0) - 
          (i > 0 && j > 0 ? II[i - 1][j - 1] : 0);
      }
    }

    var x1El     = root.querySelector('#sim-ep0803_x1');
    var y1El     = root.querySelector('#sim-ep0803_y1');
    var x2El     = root.querySelector('#sim-ep0803_x2');
    var y2El     = root.querySelector('#sim-ep0803_y2');
    var vTl      = root.querySelector('#sim-ep0803_v_tl');
    var vBr      = root.querySelector('#sim-ep0803_v_br');
    var gridI    = root.querySelector('#sim-ep0803_gridI');
    var gridII   = root.querySelector('#sim-ep0803_gridII');
    var formulaEl= root.querySelector('#sim-ep0803_formula');
    var verifyEl = root.querySelector('#sim-ep0803_verify');
    var badge    = root.querySelector('#sim-ep0803_badge');

    var CELL = 34, HEAD = 20;

    function cellDiv(text, size, extraStyle){
      var d = document.createElement('div');
      d.style.cssText = 'display:flex; align-items:center; justify-content:center; font-family:monospace; font-size:' + size + 'px;' + extraStyle;
      d.textContent = text;
      return d;
    }

    function clampAndRender(changed){
      var x1 = +x1El.value, y1 = +y1El.value, x2 = +x2El.value, y2 = +y2El.value;
      if (changed === 'x1' && x1 > x2) x2El.value = x1;
      if (changed === 'x2' && x2 < x1) x1El.value = x2;
      if (changed === 'y1' && y1 > y2) y2El.value = y1;
      if (changed === 'y2' && y2 < y1) y1El.value = y2;
      render();
    }

    function render(){
      var x1 = +x1El.value, y1 = +y1El.value, x2 = +x2El.value, y2 = +y2El.value;
      vTl.textContent = '(' + x1 + ',' + y1 + ')';
      vBr.textContent = '(' + x2 + ',' + y2 + ')';

      var sit;
      if (x1 === x2 && y1 === y2) sit = 'Pixel Único';
      else if (x1 === 0 && y1 === 0) sit = 'Desde a Origem';
      else if (x1 === 0) sit = 'Borda Esquerda';
      else if (y1 === 0) sit = 'Borda Superior';
      else sit = 'Interno';

      badge.textContent = sit;

      // Grade I
      gridI.style.gridTemplateColumns = HEAD + 'px repeat(' + N + ',' + CELL + 'px)';
      gridI.style.gridTemplateRows = HEAD + 'px repeat(' + N + ',' + CELL + 'px)';
      gridI.innerHTML = '';
      gridI.appendChild(cellDiv('', 10, 'color:#8a8371;'));
      for (var c = 0; c < N; c++) gridI.appendChild(cellDiv(c, 10, 'color:#8a8371; font-weight:700;'));
      
      for (var r = 0; r < N; r++){
        gridI.appendChild(cellDiv(r, 10, 'color:#8a8371; font-weight:700;'));
        for (var c = 0; c < N; c++){
          var dentro = (r >= y1 && r <= y2 && c >= x1 && c <= x2);
          gridI.appendChild(cellDiv(I[r][c], 12, 'border-radius:4px; transition:all 0.15s ease;' +
            (dentro 
              ? 'background:#f1ead7; border:2px solid #26241d; font-weight:700; color:#26241d;'
              : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;')));
        }
      }

      // Grade II (5x5 dados)
      var M = N + 1;
      gridII.style.gridTemplateColumns = HEAD + 'px repeat(' + M + ',' + CELL + 'px)';
      gridII.style.gridTemplateRows = HEAD + 'px repeat(' + M + ',' + CELL + 'px)';
      gridII.innerHTML = '';
      gridII.appendChild(cellDiv('', 10, 'color:#8a8371;'));
      for (var c2 = 0; c2 < M; c2++) gridII.appendChild(cellDiv(c2 - 1, 10, 'color:#8a8371; font-weight:700;'));

      var t1 = [y2 + 1, x2 + 1];
      var t2 = [y2 + 1, x1];
      var t3 = [y1, x2 + 1];
      var t4 = [y1, x1];

      function styleFor(r2, c2){
        var isVirtual = (r2 === 0 || c2 === 0);
        var base = isVirtual
          ? 'border-radius:4px; background:#fafaf7; border:1px dashed #e4dcc8; color:#8a8371;'
          : 'border-radius:4px; background:#fafaf7; border:1px solid #e4dcc8; color:#26241d;';
        
        function match(t, color, tcolor){
          if (r2 === t[0] && c2 === t[1]) {
            return 'border-radius:4px; font-weight:700; background:' + color + '; border:2px solid ' + tcolor + '; color:#ffffff;';
          }
          return null;
        }

        return match(t1, '#2980b9', '#1c5d85') || 
               match(t2, '#d35400', '#a04000') ||
               match(t3, '#d35400', '#a04000') || 
               match(t4, '#27ae60', '#1e8449') || base;
      }

      for (var r2 = 0; r2 < M; r2++){
        gridII.appendChild(cellDiv(r2 - 1, 10, 'color:#8a8371; font-weight:700;'));
        for (var c2 = 0; c2 < M; c2++){
          var val = (r2 === 0 || c2 === 0) ? 0 : II[r2 - 1][c2 - 1];
          gridII.appendChild(cellDiv(val, 12, styleFor(r2, c2)));
        }
      }

      function term(y, x){ return (y < 0 || x < 0) ? 0 : II[y][x]; }
      var a = term(y2, x2), b = term(y2, x1 - 1), c3 = term(y1 - 1, x2), d = term(y1 - 1, x1 - 1);
      var S = a - b - c3 + d;

      formulaEl.innerHTML =
        'S = II(' + y2 + ',' + x2 + ') &minus; II(' + y2 + ',' + (x1 - 1) + ') &minus; II(' + (y1 - 1) + ',' + x2 + ') + II(' + (y1 - 1) + ',' + (x1 - 1) + ')<br>' +
        'S = ' + a + ' &minus; ' + b + ' &minus; ' + c3 + ' + ' + d + ' = <b>' + S + '</b>';

      var direta = 0;
      for (var rr = y1; rr <= y2; rr++){
        for (var cc = x1; cc <= x2; cc++){
          direta += I[rr][cc];
        }
      }

      if (direta === S) {
        verifyEl.style.borderColor = '#a3e4d7';
        verifyEl.style.background  = '#eafaf1';
        verifyEl.style.color       = '#04342C';
      } else {
        verifyEl.style.borderColor = '#f5b7b1';
        verifyEl.style.background  = '#fdecea';
        verifyEl.style.color       = '#c0392b';
      }

      verifyEl.innerHTML = '&#10004; Verificação (Soma Direta dos Pixels) = ' + direta + (direta === S ? ' &rarr; Bate com S' : ' &rarr; Erro');
    }

    x1El.addEventListener('input', function(){ clampAndRender('x1'); });
    y1El.addEventListener('input', function(){ clampAndRender('y1'); });
    x2El.addEventListener('input', function(){ clampAndRender('x2'); });
    y2El.addEventListener('input', function(){ clampAndRender('y2'); });

    root.querySelectorAll('button[data-preset]').forEach(function(btn){
      btn.addEventListener('click', function(){
        var p = btn.getAttribute('data-preset').split(',').map(Number);
        x1El.value = p[0]; y1El.value = p[1]; x2El.value = p[2]; y2El.value = p[3];
        render();
      });
    });

    render();
  }

  function tryInitSim08Ep03(){
    var root = document.getElementById('sim-ep0803');
    if (root) initSim08Ep03(root); else setTimeout(tryInitSim08Ep03, 200);
  }
  tryInitSim08Ep03();
})();
</script>
""")

**Figura 8.3:** Simulador EP08_03: Soma Rectangular en O(1) — Múltiples Situaciones de Borde


In [ ]:
%%writefile EP08_03.py
# Código Python

In [ ]:
TestSuite("EP08_03.py").run()

### EP08_04 🟢 IoU y Supresión de No Máximos (NMS)

La figura de esta sección mostró el efecto de la Supresión de No Máximos sobre un conjunto de cajas producidas por un detector tipo *sliding window*: múltiples detecciones redundantes por objeto se redujeron a una única caja por objeto. Se le ha encargado reimplementar, byte a byte, las dos funciones que produjeron ese resultado — `calcular_iou` y `supresion_no_maximos` — para confirmar, con sus propias manos, exactamente los números que el capítulo presentó.

#### 📋 Directrices de Implementación

1. **Entrada:** Leer el entero $N$ (número de cajas) y el real $\tau$ (umbral de IoU). A continuación, leer $N$ líneas, cada una con cinco reales $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.
2. **Intersección sobre Unión:** Para dos cajas $A$ y $B$,
   $$
   \mathrm{IoU}(A,B) = \frac{\text{área}(A \cap B)}{\text{área}(A \cup B)},
   $$
   con área de intersección nula cuando las cajas no se superponen.
3. **Algoritmo de NMS** (exactamente como se describe en el capítulo):
   
   a. Ordenar las cajas por `score` descendente (los empates mantienen el orden de lectura original).

   b. Seleccionar la caja de mayor puntuación entre las restantes; añadirla a la salida y eliminarla de la lista.

   c. Descartar, de la lista restante, **todas** las cajas cuyo IoU con la caja seleccionada sea **mayor o igual** a $\tau$ — solo las cajas con $\mathrm{IoU} < \tau$ permanecen como candidatas.
   
   d. Repetir (b)–(c) hasta que la lista de restantes esté vacía.

4. **Salida:** Para cada caja mantenida, en el orden en que fue seleccionada, imprimir su índice original (posición de lectura, a partir de $0$) y su `score`, con 2 decimales. Al final, imprimir `Total mantenidas: X`.

#### 📌 Restricciones Computacionales

* **Atención al sentido del umbral:** al contrario de lo que se podría suponer, una caja es **suprimida** cuando $\mathrm{IoU} \ge \tau$ (no solo cuando $\mathrm{IoU} > \tau$) — seguir exactamente ese criterio, el mismo del código de referencia del capítulo.
* **Índices originales:** la salida hace referencia a la posición de lectura de cada caja en la entrada, no a su posición después de la ordenación por `score`.
* **Área sin suma de 1 píxel:** usar área $= (x_{max}-x_{min}) \times (y_{max}-y_{min})$, exactamente como en el capítulo (sin el ajuste "+1" a veces usado en otras convenciones).

#### 🧠 Fundamentación Teórica

| Elemento | Papel en el postprocesamiento |
|---|---|
| IoU | Cuantifica la superposición espacial entre dos cajas delimitadoras |
| *Sliding window* (Haar Cascade) | Produce típicamente varias detecciones superpuestas para el mismo objeto, en posiciones y escalas cercanas |
| Umbral $\tau$ | Controla la agresividad de la supresión: demasiado bajo fusiona objetos cercanos; demasiado alto deja pasar redundancias |
| Ordenación por `score` | Garantiza que, entre cajas redundantes, la de mayor confianza siempre sobrevive |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $N$ y real $\tau$.
* Siguientes $N$ líneas: cinco reales $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.

**Salida:**

* Una línea por caja mantenida, en el orden de selección: `índice score` (score con 2 decimales).
* Última línea: `Total mantenidas: X`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 5 0.4<br>50 50 150 150 0.90<br>60 55 155 145 0.75<br>58 60 160 150 0.60<br>300 300 400 420 0.95<br>310 305 395 415 0.70 | 3 0.95<br>0 0.90<br>Total mantenidas: 2 | Exactamente el ejemplo de la figura del capítulo: 5 cajas redundantes (2 objetos) se convierten en 2 detecciones finales. El IoU entre la 1.ª y la 2.ª cajas es $\approx 0{,}775$, muy por encima de $\tau=0{,}4$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0804" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0804 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0804 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0804 button:hover { background: #e8dfcf; }
  #sim-ep0804 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0804_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0804_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP08_04: IoU y Supresión de No-Máximos (NMS)</span>
  <span class="sim-ep0804_pill">Supresión si IoU &ge; &tau;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0804_panel" style="margin-bottom:14px;">
    <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:12px;">
      
      <div>
        <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Desplazamiento de la Candidata (dx)</label>
          <span id="sim-ep0804_dx_v" style="font-family:monospace; font-weight:700; color:#26241d;">3</span>
        </div>
        <input id="sim-ep0804_dx" type="range" min="0" max="10" step="1" value="3">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Umbral (&tau;)</label>
          <span id="sim-ep0804_tau_v" style="font-family:monospace; font-weight:700; color:#26241d;">0.40</span>
        </div>
        <input id="sim-ep0804_tau" type="range" min="0.1" max="0.9" step="0.05" value="0.4">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      La caja azul (puntaje mayor) ya fue seleccionada. Ajuste la superposición y el umbral &tau; para verificar la supresión de la caja roja (candidata).
    </div>
  </div>

  <!-- Canvas Visual de Caixas Delimitadoras -->
  <div class="sim-ep0804_panel" style="position:relative; width:100%; height:160px; margin-bottom:14px; overflow:hidden;">
    <div id="sim-ep0804_boxA" style="position:absolute; border:2px solid #2980b9; background:rgba(41,128,185,0.20); border-radius:4px; transition:all 0.15s ease;"></div>
    <div id="sim-ep0804_boxB" style="position:absolute; border:2px solid #c0392b; background:rgba(192,57,43,0.20); border-radius:4px; transition:all 0.15s ease;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0804_debug" class="sim-ep0804_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep04(root){
    if (!root || root.dataset.sim08Ep04Init) return;
    root.dataset.sim08Ep04Init = "1";

    var dxEl   = root.querySelector('#sim-ep0804_dx');
    var dxvEl  = root.querySelector('#sim-ep0804_dx_v');
    var tauEl  = root.querySelector('#sim-ep0804_tau');
    var tauvEl = root.querySelector('#sim-ep0804_tau_v');
    var boxA   = root.querySelector('#sim-ep0804_boxA');
    var boxB   = root.querySelector('#sim-ep0804_boxB');
    var dbg    = root.querySelector('#sim-ep0804_debug');

    var ESCALA = 10;
    var A = {x1: 5, y1: 3, x2: 15, y2: 13};

    function iou(a, b){
      var ix1 = Math.max(a.x1, b.x1), iy1 = Math.max(a.y1, b.y1);
      var ix2 = Math.min(a.x2, b.x2), iy2 = Math.min(a.y2, b.y2);
      var iw  = Math.max(0, ix2 - ix1), ih = Math.max(0, iy2 - iy1);
      var inter = iw * ih;
      var areaA = (a.x2 - a.x1) * (a.y2 - a.y1);
      var areaB = (b.x2 - b.x1) * (b.y2 - b.y1);
      return inter / (areaA + areaB - inter);
    }

    function render(){
      var dx  = parseInt(dxEl.value, 10);
      var tau = parseFloat(tauEl.value);

      dxvEl.textContent  = dx;
      tauvEl.textContent = tau.toFixed(2);

      var B = {x1: 5 + dx, y1: 3 + dx * 0.4, x2: 15 + dx, y2: 13 + dx * 0.4};

      boxA.style.left   = (A.x1 * ESCALA) + 'px';
      boxA.style.top    = (A.y1 * ESCALA) + 'px';
      boxA.style.width  = ((A.x2 - A.x1) * ESCALA) + 'px';
      boxA.style.height = ((A.y2 - A.y1) * ESCALA) + 'px';

      boxB.style.left   = (B.x1 * ESCALA) + 'px';
      boxB.style.top    = (B.y1 * ESCALA) + 'px';
      boxB.style.width  = ((B.x2 - B.x1) * ESCALA) + 'px';
      boxB.style.height = ((B.y2 - B.y1) * ESCALA) + 'px';

      var val = iou(A, B);
      var suprimida = val >= tau;

      if (suprimida) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'IoU(A,B) = ' + val.toFixed(4) + '  |  \u03C4 = ' + tau.toFixed(2) + '  \u2192  Candidata (Vermelha) ' + 
        (suprimida ? 'SUPRIMIDA (IoU \u2265 \u03C4)' : 'MANTIDA (IoU < \u03C4)');
    }

    dxEl.addEventListener('input', render);
    tauEl.addEventListener('input', render);
    render();
  }

  function tryInitSim08Ep04(){
    var root = document.getElementById('sim-ep0804');
    if (root) initSim08Ep04(root); else setTimeout(tryInitSim08Ep04, 200);
  }
  tryInitSim08Ep04();
})();
</script>
""")

**Figura 8.4:** Simulador EP08_04: IoU y Supresión de No-Máximos


In [ ]:
%%writefile EP08_04.py
# Código Python

In [ ]:
TestSuite("EP08_04.py").run()

### EP08_05 🟡 Etiquetado de Componentes Conectados: Segmentación de Instancias

El ejemplo de segmentación clásica de este capítulo separó "instancias" de monedas simplemente por su desconexión espacial en la máscara binaria resultante de la umbralización de Otsu. Esa etapa final — etiquetar cada componente conectado con un identificador de instancia — es exactamente lo que se te ha encargado implementar aquí, desde cero, sobre una máscara binaria ya preparada (0 = fondo, 1 = objeto), como si fuera una reimplementación manual de `cv2.connectedComponents`.

Este ejercicio también expone, de forma muy concreta, la limitación discutida en el capítulo: el resultado depende enteramente de cómo se define "vecindad" entre píxeles — y, como verás en el segundo ejemplo, dos píxeles en diagonal pueden considerarse la misma instancia o instancias diferentes, dependiendo exclusivamente de la **conectividad** elegida, no de ninguna noción semántica de objeto.

#### 📋 Directrices de Implementación

1. **Entrada:** Leer las dimensiones $H \times W$ de la máscara binaria y sus $H \times W$ valores ($0$ o $1$).
2. **Conectividad:** Leer el entero $c \in \{4, 8\}$. En la conectividad $4$, los vecinos de $(i,j)$ son $(i{-}1,j)$, $(i{+}1,j)$, $(i,j{-}1)$ y $(i,j{+}1)$. En la conectividad $8$, se suman las cuatro diagonales: $(i{-}1,j{-}1)$, $(i{-}1,j{+}1)$, $(i{+}1,j{-}1)$ y $(i{+}1,j{+}1)$.
3. **Descubrimiento de componentes:** Recorriendo la máscara en un barrido línea a línea, de izquierda a derecha y de arriba hacia abajo, siempre que se encuentre un píxel de valor $1$ aún sin etiqueta, este inicia un **nuevo componente**: asígnale la siguiente etiqueta disponible (el primer componente descubierto recibe la etiqueta $1$, el segundo la etiqueta $2$, y así sucesivamente) y propaga esa misma etiqueta a todos los píxeles de valor $1$ alcanzables desde él mediante una cadena de vecinos (de acuerdo con la conectividad elegida) — por búsqueda en anchura, en profundidad, o *union-find*, a tu elección.
4. **Píxeles de fondo:** permanecen con etiqueta $0$ y no pertenecen a ninguna instancia.
5. **Salida:** Primero, imprimir el mapa de etiquetas completo — $H$ líneas con $W$ enteros cada una. A continuación, para cada etiqueta $\ell$ de $1$ a $K$ (en el orden de descubrimiento), imprimir `Instancia l: A píxeles`, donde $A$ es la cantidad de píxeles con esa etiqueta. Por último, imprimir `Total de instancias: K`.

#### 📌 Restricciones Computacionales

* **Orden de descubrimiento = orden de barrido:** las etiquetas se numeran en el orden en que cada nuevo componente se encuentra mediante el barrido línea a línea, no por tamaño ni posición.
* **Conectividad explícita:** dos píxeles de valor $1$ solo pertenecen a la misma instancia si existe una cadena de vecinos **de acuerdo con $c$** que los conecte entre sí — no uses la conectividad opuesta por error.
* **Máscara binaria pura:** todos los valores de entrada son exactamente $0$ o $1$.

#### 🧠 Fundamentación Teórica

| Elemento | Papel en la segmentación clásica de instancias |
|---|---|
| Umbralización (Otsu, Cap. 4) | Etapa anterior que produce la máscara binaria a partir de la imagen de intensidad |
| Componente conectado | Cada instancia se define **solo** por la conectividad espacial de los píxeles de objeto, sin ninguna noción de forma, clase o apariencia |
| Conectividad 4 vs. 8 | Parámetro que altera el resultado: bajo conectividad 8, dos blobs unidos solo en diagonal se convierten en una única instancia |
| Limitación central | La técnica fusiona instancias que se tocan o se superponen (aunque sean objetos claramente distintos), pues no hay noción de "objeto" — solo de "región conectada" |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $H$ y $W$.
* Siguientes $H$ líneas: $W$ enteros ($0$ o $1$) cada una.
* Última línea: Entero $c$ ($4$ o $8$).

**Salida:**

* $H$ líneas con $W$ enteros cada una (el mapa de etiquetas).
* Una línea por instancia, en el orden de descubrimiento: `Instancia l: A píxeles`.
* Última línea: `Total de instancias: K`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 6 6<br>0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 1 1<br>0 0 0 0 1 1<br>8 | 0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 2 2<br>0 0 0 0 2 2<br>Instancia 1: 4 píxeles<br>Instancia 2: 4 píxeles<br>Total de instancias: 2 | Dos bloques $2\times2$ claramente separados: el resultado es el mismo bajo conectividad 4 u 8. |
| 2 2<br>1 0<br>0 1<br>8 | 1 0<br>0 1<br>Instancia 1: 2 píxeles<br>Total de instancias: 1 | Bajo conectividad 8, los dos píxeles en diagonal pertenecen a la **misma** instancia. Repite este ejemplo con $c=4$: el resultado pasa a ser 2 instancias de 1 píxel cada una — puramente por el cambio de conectividad, sin ninguna diferencia en la máscara. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0805" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0805 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0805 button { font-size: 11px; padding: 6px 16px; border-radius: 20px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0805 button:hover { background: #e8dfcf; }
  #sim-ep0805 button.sim-ep0805_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0805_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0805_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP08_05: Componentes Conexos (Conectividad 4 vs. 8)</span>
  <span class="sim-ep0805_pill">Misma Máscara &rarr; Etiquetas Diferentes</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles de Seleção de Conectividade -->
  <div class="sim-ep0805_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      La misma máscara (dos píxeles en diagonal) &mdash; cambia la conectividad y observa cómo cambian el número de instancias y los colores de las etiquetas.
    </div>

    <div style="display:flex; gap:8px; justify-content:center;">
      <button id="sim-ep0805_c4">Conectividad 4</button>
      <button id="sim-ep0805_c8" class="sim-ep0805_active">Conectividad 8</button>
    </div>
  </div>

  <!-- Exibição da Grade 2x2 -->
  <div class="sim-ep0805_panel" style="margin-bottom:14px; display:flex; justify-content:center;">
    <div id="sim-ep0805_grid" style="display:grid; grid-template-columns:repeat(2, 56px); gap:6px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0805_debug" class="sim-ep0805_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep05(root){
    if (!root || root.dataset.sim08Ep05Init) return;
    root.dataset.sim08Ep05Init = "1";

    var mask = [[1, 0], [0, 1]];
    var conect = 8;
    var CORES = ['#eafaf1', '#fdecea'];
    var BORDAS = ['#a3e4d7', '#f5b7b1'];
    var TEXTOS = ['#04342C', '#c0392b'];

    var btn4   = root.querySelector('#sim-ep0805_c4');
    var btn8   = root.querySelector('#sim-ep0805_c8');
    var gridEl = root.querySelector('#sim-ep0805_grid');
    var dbg    = root.querySelector('#sim-ep0805_debug');

    function rotula(){
      var H = mask.length, W = mask[0].length;
      var labels = [[0, 0], [0, 0]];
      var atual = 0;
      var viz4 = [[-1, 0], [1, 0], [0, -1], [0, 1]];
      var viz8 = viz4.concat([[-1, -1], [-1, 1], [1, -1], [1, 1]]);
      var viz = conect === 8 ? viz8 : viz4;

      for (var i = 0; i < H; i++){
        for (var j = 0; j < W; j++){
          if (mask[i][j] === 1 && labels[i][j] === 0){
            atual++;
            var fila = [[i, j]];
            labels[i][j] = atual;
            while (fila.length){
              var pos = fila.pop();
              var r = pos[0], c = pos[1];
              for (var k = 0; k < viz.length; k++){
                var nr = r + viz[k][0], nc = c + viz[k][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W && mask[nr][nc] === 1 && labels[nr][nc] === 0){
                  labels[nr][nc] = atual;
                  fila.push([nr, nc]);
                }
              }
            }
          }
        }
      }
      return {labels: labels, k: atual};
    }

    function estiloBotoes(){
      btn4.classList.toggle('sim-ep0805_active', conect === 4);
      btn8.classList.toggle('sim-ep0805_active', conect === 8);
    }

    function render(){
      var res = rotula();
      gridEl.innerHTML = '';

      for (var i = 0; i < 2; i++){
        for (var j = 0; j < 2; j++){
          var d = document.createElement('div');
          var lab = res.labels[i][j];
          var estilo = 'width:56px; height:56px; display:flex; align-items:center; justify-content:center; border-radius:8px; font-family:monospace; font-weight:700; font-size:13px; transition:all 0.15s ease;';
          
          if (lab === 0){
            estilo += 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;';
          } else {
            var idx = (lab - 1) % 2;
            estilo += 'background:' + CORES[idx] + '; border:2px solid ' + BORDAS[idx] + '; color:' + TEXTOS[idx] + ';';
          }

          d.style.cssText = estilo;
          d.textContent = mask[i][j] + (lab ? ' (r' + lab + ')' : '');
          gridEl.appendChild(d);
        }
      }

      estiloBotoes();

      if (res.k === 1) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#26241d';
      }

      dbg.textContent = 'Conectividad = ' + conect + '  \u2192  ' + res.k + ' Instância(s) Encontrada(s)';
    }

    btn4.addEventListener('click', function(){ conect = 4; render(); });
    btn8.addEventListener('click', function(){ conect = 8; render(); });

    render();
  }

  function tryInitSim08Ep05(){
    var root = document.getElementById('sim-ep0805');
    if (root) initSim08Ep05(root); else setTimeout(tryInitSim08Ep05, 200);
  }
  tryInitSim08Ep05();
})();
</script>
""")

**Figura 8.5:** Simulador EP08_05: Etiquetado de Componentes Conectados — Conectividad 4 vs. 8


In [ ]:
%%writefile EP08_05.py
# Código Python

In [ ]:
TestSuite("EP08_05.py").run()

### EP08_06 🟡 *Bounding Boxes*, Centroides y Propiedades de Instancias con `mm.measure`

En el ejercicio anterior (**EP08_05**), se puede observar cómo la segmentación por componentes conectados etiqueta regiones binarias contiguas para separar instancias. Sin embargo, para tareas de detección, seguimiento y análisis cuantitativo de objetos, el simple mapa de etiquetas no es suficiente. Se vuelve necesario extraer **métricas espaciales y geométricas** que caractericen cada instancia individualmente.

Este EP se centra en el cálculo y la extracción automática de las propiedades fundamentales de visión por computadora para cada componente conectado encontrado en la máscara binaria, utilizando el método nativo `mm.measure(img)` de la biblioteca `morph`:

1. **Caja Delimitadora (*Bounding Box*):** El rectángulo más pequeño alineado con los ejes que envuelve completamente la instancia, definido por su esquina superior izquierda $(x, y)$, ancho $w$ y alto $h$.
2. **Centroide Geométrico $(\bar{x}, \bar{y})$:** El centro de masa de la instancia en la cuadrícula discreta, equivalente a los momentos espaciales de primer orden $M_{10}/M_{00}$ y $M_{01}/M_{00}$.
3. **Área Geométrica del Contorno ($A$):** El área delimitada por el contorno de la instancia calculada mediante `mm.contourArea(c)`.



#### 📋 Directrices de Implementación

1. **Entrada:** Leer las dimensiones $H \times W$ de la máscara binaria, los $H \times W$ valores ($0$ o $1$) y el parámetro de conectividad $c \in \{4, 8\}$.
2. **Extracción Automática con `mm.measure`:** Pasar la imagen binarizada a la función `mm.measure(img_bin)`, que extrae los contornos OpenCV y devuelve una lista de diccionarios que contienen las propiedades geométricas de cada instancia.
3. **Propiedades Devueltas:** Para cada diccionario $m$ de la lista devuelta por `medidas = mm.measure(img_bin)`:
   * **Área (`area`):** Valor numérico del área geométrica del contorno `mm.contourArea(c)`.
   * **Bounding Box (`bbox`):** Tupla $(x, y, w, h)$ que representa la esquina superior izquierda, el ancho y el alto.
   * **Centroide (`center`):** Tupla $(c_x, c_y)$ con las coordenadas del centro de masa $M_{10}/M_{00}$ y $M_{01}/M_{00}$. Formatear con **dos decimales**.
4. **Salida:** Para cada instancia $1, \dots, K$ encontrada (ordenada por orden de descubrimiento/posición en la imagen), imprimir una línea que contenga sus propiedades. Finalmente, imprimir el número total de instancias.
   - Para ordenar, usar `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`.



#### 🧠 Fundamentación Teórica

| Propiedad en `mm.measure` | Cálculo Matemático / Lógica Discreta | Aplicación Práctica en Visión |
| --- | --- | --- |
| **`bbox` (OpenCV)** | $[x, y, w, h] = [\min(c), \min(r), \Delta c + 1, \Delta r + 1]$ | Formato clásico de OpenCV. *Nota: redes como YOLO convierten este rectángulo a $(c_x, c_y, w, h)$ normalizado.* |
| **`center`** | $\bar{x} = \frac{M_{10}}{M_{00}}, \quad \bar{y} = \frac{M_{01}}{M_{00}}$ | Centro de masa exacto de la máscara (usado en seguimiento y análisis de trayectoria). |
| **`area`** | $A = \text{contourArea}(C)$ (Fórmula del Polígono) | Métrica continua de la superficie del objeto. |


#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $H$ y $W$.
* Siguientes $H$ líneas: $W$ enteros ($0$ o $1$) cada una.
* Última línea: Entero $c$ ($4$ o $8$).

**Salida:**

* Una línea por instancia en el orden de descubrimiento:
`Instancia l: Area=A, BBox=(x,y,w,h), Centroide=(cx,cy)`
* Última línea: `Total de instancias: K`.



#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 6 6<br>0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 1 1<br>0 0 0 0 1 1<br>8 | Instancia 1: Area=1.0, BBox=(1,1,2,2), Centroide=(1.50,1.50)<br>Instancia 2: Area=1.0, BBox=(4,4,2,2), Centroide=(4.50,4.50)<br>Total de instancias: 2 | Bloques $2\times2$ alineados. El cálculo del área geométrica del contorno resulta en $1.0$. El centroide del bloque en las columnas 1–2 y filas 1–2 es exactamente $(1.50,\,1.50)$. |
| 4 6<br>0 0 0 0 0 0<br>0 1 1 1 1 0<br>0 0 0 1 0 0<br>0 0 0 0 0 0<br>4 | Instancia 1: Area=2.0, BBox=(1,1,4,2), Centroide=(2.40,1.20)<br>Total de instancias: 1 | Objeto asimétrico en forma de "T" invertida. El área geométrica del contorno es $2.0$. El centroide refleja la distribución de los píxeles del objeto. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0806" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulador EP08_06: Métricas Morfológicas Nativas (mm.measure)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">OpenCV Contour& Momentos</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:14px;flex-wrap:wrap;gap:10px;">
      <div>
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ACCIÓN</div>
        <button id="ep0806_btnRandom" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generar Nuevas Instancias Binarias</button>
      </div>
      <div style="font-size:11px;color:#8a8371;font-family:monospace;">
        <span style="font-weight:700;color:#26241d;">Parámetro de precisión (approxPolyDP):</span> precision = 0.01
      </div>
    </div>

    <!-- Container da Matriz de Píxeis -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">MAPA DE ETIQUETAS DE INSTANCIAS</div>
      <div id="ep0806_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <!-- Tabela de Métricas do mm.measure -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MÉTRICAS EXTRAÍDAS POR MM.MEASURE</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">área</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perímetro</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">centro (cx, cy)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularidad</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidez</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vértices</th>
            </tr>
          </thead>
          <tbody id="ep0806_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0806Init) return;
    root.dataset.ep0806Init = "1";

    var H = 10, W = 22;
    var mask = [], labels = [], metrics = [];
    var colors = ['#ffffff', '#7ee7c6', '#fca5a5', '#fde047', '#93c5fd', '#c084fc', '#f472b6'];

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [];
            var queue = [[r, c]];
            visited[r][c] = true;

            while (queue.length > 0) {
              var curr = queue.shift();
              var cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);

              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true;
                    queue.push([nr, nc]);
                  }
                }
              }
            }

            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) {
                borderPts.push([pc, pr]);
              }
            });

            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;

            borderPts.sort(function(a, b) {
              return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx);
            });

            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1];
        area -= contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length;
      var m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }

      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else {
          return [pts[0], pts[end]];
        }
      }
      return rdp(contour, epsilon);
    }

    function measureJS(mat) {
      var blobs = cv2_findContours(mat);
      var res = [];
      labels = Array.from({length: H}, function(){ return Array(W).fill(0); });

      blobs.forEach(function(item, idx) {
        var contour = item.contour;
        var pixels = item.pixels;
        var labelId = idx + 1;

        pixels.forEach(function(p){ labels[p[0]][p[1]] = labelId; });

        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;

        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);

        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;

        var poly = cv2_approxPolyDP(contour, 0.01);

        res.push({
          id: labelId,
          area: area,
          perimeter: per,
          cx: moments.cx,
          cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0,
          vertices: poly.length
        });
      });

      res.sort(function(a, b) {
        if (a.y !== b.y) return a.y - b.y;
        return a.x - b.x;
      });

      res.forEach(function(m, i) { m.id = i + 1; });
      return res;
    }

    function render() {
      var gridContainer = root.querySelector('#ep0806_grid_container');
      gridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var l = labels[r][c];
          var cell = document.createElement('div');
          var bg = l === 0 ? '#ffffff' : colors[(l % (colors.length - 1)) + 1];
          var fg = l === 0 ? '#8a8371' : '#26241d';
          cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;font-family:monospace;user-select:none;background:' + bg + ';color:' + fg + ';';
          cell.textContent = l;
          grid.appendChild(cell);
        }
      }
      gridContainer.appendChild(grid);

      var tbody = root.querySelector('#ep0806_tbody');
      tbody.innerHTML = '';

      if (metrics.length === 0) {
        tbody.innerHTML = '<tr><td colspan="8" style="padding:12px;color:#8a8371;text-align:center;">Nenhuma instância binária encontrada.</td></tr>';
        return;
      }

      metrics.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        var centerStr = '(' + m.cx.toFixed(2) + ', ' + m.cy.toFixed(2) + ')';
        var bboxStr = '(' + m.x + ', ' + m.y + ', ' + m.w + ', ' + m.h + ')';

        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + centerStr + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxStr + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        tbody.appendChild(tr);
      });
    }

    function generate() {
      mask = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var numObj = Math.floor(Math.random() * 2) + 2;

      for (var o = 0; o < numObj; o++) {
        var w = Math.floor(Math.random() * 3) + 3;
        var h = Math.floor(Math.random() * 3) + 3;
        var sr = Math.floor(Math.random() * (H - h));
        var sc = Math.floor(Math.random() * (W / numObj - w)) + Math.floor(o * (W / numObj));

        for (var r = 0; r < h; r++) {
          for (var c = 0; c < w; c++) {
            if (Math.random() > 0.15) mask[sr + r][sc + c] = 1;
          }
        }
      }

      metrics = measureJS(mask);
      render();
    }

    root.querySelector('#ep0806_btnRandom').addEventListener('click', generate);
    generate();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0806');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.6:** Simulador EP08_06: Extracción de *Bounding Boxes*, Centroides y Propiedades con *mm.measure*


In [ ]:
%%writefile EP08_06.py
# Código Python

In [ ]:
TestSuite("EP08_06.py").run()

### EP08_07 🟡 Eliminación de Ruido Sal y Pimienta y Medición de Objetos

En este ejercicio, usted aplicará filtrado morfológico para limpiar una imagen binaria corrompida por ruido del tipo **sal y pimienta** (píxeles aislados de valor `1` en el fondo y `0` en el interior de los objetos). Después de la limpieza, el programa debe extraer las mediciones geométricas de los componentes conectados restantes, ordenarlos y mostrar la tabla final de métricas.

#### 📋 Directrices de Implementación

1. **Entrada:** leer dos enteros $H$ y $W$ (alto y ancho de la imagen) en la primera línea y, a continuación, las $H$ líneas con la matriz binaria que contiene píxeles `0` y `1` separados por espacio.

2. **Filtrado Morfológico:** aplicar encadenamiento de **Apertura** (para eliminar el ruido sal en el fondo) seguido de **Cierre** (para rellenar el ruido pimienta dentro de los objetos) con elemento estructurante $3 \times 3$.

3. **Impresión de la Imagen Limpia:** imprimir la matriz resultante en valores `0` y `1` separados por espacio.

4. **Mediciones Geométricas:** para cada objeto identificado en la matriz limpia, extraer:
* `id`: identificador numérico secuencial (reasignado después de la ordenación);

* `area`: área calculada mediante contorno (`cv2.contourArea`);

* `perimeter`: perímetro del contorno (`cv2.arcLength`);

* `cx`, `cy`: centro de masa (centroide mediante `cv2.moments`);

* `x`, `y`, `w`, `h`: coordenadas del rectángulo delimitador (`cv2.boundingRect`);

* `circularity`: circularidad dada por $\frac{4 \pi \cdot \text{área}}{\text{perímetro}^2}$;
* `solidity`: solidez dada por la razón $\frac{\text{área}}{\text{área del casco convexo}}$;
* `vertices`: número de vértices aproximado del polígono (`cv2.approxPolyDP` con $\epsilon = 0.02 \times \text{perímetro}$).

5. **Ordenación y Salida:** ordenar los objetos en orden creciente por la posición $X$ del rectángulo delimitador (`bbox[0]`); en caso de empate, usar la posición $Y$ (`bbox[1]`). Reasignar los `id`s de $1$ a $N$ e imprimir la tabla formateada.
   - Para ordenar, usar `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `medidas = mm.measure(img)`.

#### 📌 Restricciones y Reglas de Ordenación

* **Regla de Ordenación de los Objetos:**
```python
medidas.sort(key=lambda m: (m['bbox'][0], m['bbox'][1]))
```

* **Diferencia de Área:** El área calculada por OpenCV (`cv2.contourArea`) mide el área del polígono continuo delimitado por los centros de los píxeles de borde, resultando en valores numéricos menores que el simple conteo discreto de píxeles `1` (`np.sum`).

#### 🧠 Fundamentación Teórica

| Operación / Métrica | Función en el Filtrado y Caracterización |
|--------------------|---------------------------------------|
| **Apertura Morfológica** ($\circ$) | Erosión seguida de dilatación: elimina ruidos brillantes aislados (*sal*). |
| **Cierre Morfológico** ($\bullet$) | Dilatación seguida de erosión: rellena pequeños huecos oscuros en el interior de los objetos (*pimienta*). |
| **`cv2.boundingRect`** | Devuelve $(x, y, w, h)$, el rectángulo más pequeño alineado a los ejes que envuelve al objeto. |
| **Circularidad y Solidez** | Describen la compacidad y la convexidad geométrica del componente. |

#### 📌 Ejemplos

| Entrada | Salida |
|---|---|
| 8 9<br>0 0 0 0 0 0 0 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 1 1<br>0 0 0 0 0 0 0 1 1<br>0 0 0 0 0 0 0 0 0 | id area perimeter cx cy x y w h circularity solidity vertices<br>1 9.0 12.0 3.5 2.0 3 1 4 3 0.79 1.000 4<br>2 4.0 8.0 7.5 5.5 7 5 2 2 0.79 1.000 4 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0807" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulador EP08_07: Morfología Conmutable (4-C / 8-C) & Métricas OpenCV</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Sal y Pimienta &rarr; Apertura &rarr; Cierre &rarr; Medición</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:260px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ETAPA DEL PROCESAMIENTO MORFOLÓGICO</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnOrig" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Ruidosa</button>
          <button id="ep0807_btnAbert" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Apertura</button>
          <button id="ep0807_btnFech" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Cierre</button>
        </div>
      </div>

      <div style="flex:1;min-width:140px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ELEMENTO ESTRUCTURANTE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnConn4" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">4-Con.</button>
          <button id="ep0807_btnConn8" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">8-Con.</button>
        </div>
      </div>

      <div style="min-width:140px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">EXHIBICIÓN DE LOS PÍXELES</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnVal" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">Valores (0/1)</button>
          <button id="ep0807_btnCor" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">Colores (P&W)</button>
        </div>
      </div>

      <div>
        <button id="ep0807_btnRandom" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generar Escenario Aleatorio</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZACIÓN DE LA MATRIZ DE PÍXELES DE ENTRADA / PROCESADA</div>
      <div id="ep0807_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABLA DE MEDICIONES DE LOS OBJETOS (CALCULADO DESPUÉS DE APERTURA Y CIERRE)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimeter</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0807_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0807Init) return;
    root.dataset.ep0807Init = "1";

    var H = 12, W = 28;
    var imgBase = [], imgRuido = [], imgAbertura = [], imgLimpa = [];
    var medidasObjetos = [];
    var etapaAtual = 'ruido', modoExibicao = 'val', modoConectividade = 4;

    var elBtnOrig = root.querySelector('#ep0807_btnOrig');
    var elBtnAbert = root.querySelector('#ep0807_btnAbert');
    var elBtnFech = root.querySelector('#ep0807_btnFech');
    var elBtnConn4 = root.querySelector('#ep0807_btnConn4');
    var elBtnConn8 = root.querySelector('#ep0807_btnConn8');
    var elBtnVal = root.querySelector('#ep0807_btnVal');
    var elBtnCor = root.querySelector('#ep0807_btnCor');
    var elBtnRandom = root.querySelector('#ep0807_btnRandom');
    var elGridContainer = root.querySelector('#ep0807_grid_container');
    var elTbody = root.querySelector('#ep0807_tbody');

    var neighbors4 = [[0,0], [-1,0], [1,0], [0,-1], [0,1]];
    var neighbors8 = [[0,0], [-1,0], [1,0], [0,-1], [0,1], [-1,-1], [-1,1], [1,-1], [1,1]];

    function dilate(mat, conn) {
      var neighbors = conn === 8 ? neighbors8 : neighbors4;
      var res = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var hit = false;
          for (var i = 0; i < neighbors.length; i++) {
            var nr = r + neighbors[i][0], nc = c + neighbors[i][1];
            if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
              if (mat[nr][nc] === 1) { hit = true; break; }
            }
          }
          res[r][c] = hit ? 1 : 0;
        }
      }
      return res;
    }

    function erode(mat, conn) {
      var neighbors = conn === 8 ? neighbors8 : neighbors4;
      var res = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var fit = true;
          for (var i = 0; i < neighbors.length; i++) {
            var nr = r + neighbors[i][0], nc = c + neighbors[i][1];
            if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
              if (mat[nr][nc] !== 1) { fit = false; break; }
            } else { fit = false; }
          }
          res[r][c] = fit ? 1 : 0;
        }
      }
      return res;
    }

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.01);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function recalcularMorfologia() {
      imgAbertura = dilate(erode(imgRuido, modoConectividade), modoConectividade);
      imgLimpa = erode(dilate(imgAbertura, modoConectividade), modoConectividade);
      medidasObjetos = measureOpenCV(imgLimpa);
      renderGrid();
      renderTabela();
    }

    function gerarCenarioAleatorio() {
      imgBase = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var numObjetos = Math.floor(Math.random() * 2) + 2; 
      var setores = [ { minC: 1, maxC: 8 }, { minC: 10, maxC: 17 }, { minC: 19, maxC: 25 } ];
      var objetoPixels = [];
      for (var o = 0; o < numObjetos; o++) {
        var setor = setores[o];
        var tipoForma = Math.floor(Math.random() * 3);
        var objW = Math.floor(Math.random() * 2) + 4, objH = Math.floor(Math.random() * 2) + 4;
        var startC = Math.floor(Math.random() * (setor.maxC - setor.minC - objW + 1)) + setor.minC;
        var startR = Math.floor(Math.random() * (H - 4 - objH + 1)) + 2;
        for (var r = 0; r < objH; r++) {
          for (var c = 0; c < objW; c++) {
            var pr = startR + r, pc = startC + c, isObj = false;
            if (tipoForma === 0) isObj = true;
            else if (tipoForma === 1) { if (r >= objH / 2 || c < objW / 2) isObj = true; }
            else if (tipoForma === 2) { if (r < objH / 2 || (c >= Math.floor(objW / 3) && c <= Math.floor(2 * objW / 3))) isObj = true; }
            if (isObj) { imgBase[pr][pc] = 1; objetoPixels.push([pr, pc]); }
          }
        }
      }
      imgRuido = JSON.parse(JSON.stringify(imgBase));
      var qtdSal = Math.floor(Math.random() * 2) + 2;
      for (var s = 0; s < qtdSal; s++) {
        var sr = Math.floor(Math.random() * (H - 2)) + 1, sc = Math.floor(Math.random() * (W - 2)) + 1;
        if (imgBase[sr][sc] === 0) imgRuido[sr][sc] = 1;
      }
      var shuffledObj = objetoPixels.filter(function(p){ return p[0] > 0 && p[0] < H-1 && p[1] > 0 && p[1] < W-1; }).sort(function() { return 0.5 - Math.random(); });
      var qtdPimenta = Math.max(1, Math.floor(shuffledObj.length * 0.10));
      for (var p = 0; p < qtdPimenta; p++) { imgRuido[shuffledObj[p][0]][shuffledObj[p][1]] = 0; }
      recalcularMorfologia();
    }

    function renderGrid(){
      var mat = imgRuido;
      if (etapaAtual === 'abertura') mat = imgAbertura;
      if (etapaAtual === 'fechamento') mat = imgLimpa;
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var val = mat[r][c], cell = document.createElement('div');
          var bg = val === 1 ? '#26241d' : '#ffffff';
          var fg = val === 1 ? '#7ee7c6' : '#8a8371';
          cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;font-family:monospace;user-select:none;background:' + bg + ';color:' + fg + ';';
          if (modoExibicao === 'val') cell.textContent = val; else cell.textContent = '';
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o processamento morfológico.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setEtapa(etapa, btn){
      [elBtnOrig, elBtnAbert, elBtnFech].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      etapaAtual = etapa; renderGrid();
    }

    function setConectividade(conn, btn){
      [elBtnConn4, elBtnConn8].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      modoConectividade = conn; recalcularMorfologia();
    }

    function setModo(modo, btn){
      [elBtnVal, elBtnCor].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      modoExibicao = modo; renderGrid();
    }

    elBtnOrig.addEventListener('click', function(){ setEtapa('ruido', elBtnOrig); });
    elBtnAbert.addEventListener('click', function(){ setEtapa('abertura', elBtnAbert); });
    elBtnFech.addEventListener('click', function(){ setEtapa('fechamento', elBtnFech); });
    elBtnConn4.addEventListener('click', function(){ setConectividade(4, elBtnConn4); });
    elBtnConn8.addEventListener('click', function(){ setConectividade(8, elBtnConn8); });
    elBtnVal.addEventListener('click', function(){ setModo('val', elBtnVal); });
    elBtnCor.addEventListener('click', function(){ setModo('cor', elBtnCor); });
    elBtnRandom.addEventListener('click', function(){ gerarCenarioAleatorio(); });

    gerarCenarioAleatorio();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0807');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.7:** Simulador EP08_07: Morfología con Conectividad Configurable y Medición


In [ ]:
%%writefile EP08_07.py
# Código Python

In [ ]:
TestSuite("EP08_07.py").run()

### EP08_08 🟡 Imagen en Niveles de Gris y Umbralización Dinámica

En este ejercicio, la imagen de entrada deja de ser estrictamente binaria (`0`/`1`) y pasa a ser una **imagen en niveles de gris ($8$ bits, $0\dots255$)**, donde los objetos poseen una intensidad media intermedia sobre un fondo oscuro ($0$), además de ruido de tipo sal y pimienta distribuido por toda la imagen.

#### 📋 Directrices de Implementación

1. **Entrada:** leer $H$ y $W$ en la primera línea, seguidos de las $H$ líneas con valores enteros de $0$ a $255$ en una matriz de $H \times W$.
2. **Preprocesamiento:**
* Aplicar un filtro de **Mediana ($3 \times 3$)** para eliminar el ruido de sal y pimienta manteniendo los bordes nítidos.
* Aplicar **Umbralización de Otsu** (o un umbral fijo $T = 60$) para binarizar la imagen limpia.


3. **Medición y Salida:** extraer el contorno de los objetos, calcular las métricas geométricas (`area`, `perimeter`, `cx`, `cy`, `x`, `y`, `w`, `h`, `circularity`, `solidity`) y ordenar los objetos por `bbox[0]` (y `bbox[1]` como criterio de desempate). Reasignar `id` de $1$ a $N$ e imprimir la tabla.
   - Para ordenar, usar `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `medidas = mm.measure(img)`.


#### 📌 Ejemplos

| Entrada | Salida |
|---|---|
| 16 32<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>(imagen binaria que contiene un cuadrado y un círculo) | id area perimeter cx cy x y w h circularity solidity vertices<br>1 16.0 16.0 8.0 8.0 6 6 5 5 0.79 1.000 4<br>2 28.3 18.8 22.5 8.0 19 5 7 7 1.00 1.000 8 |

\newpage

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0808" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulador EP08_08: Ruido Sal y Pimienta en Tonos de Gris & Medición OpenCV</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Mediana 3x3 &rarr; Binarización &rarr; Medición</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:260px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ETAPA DEL PROCESAMIENTO</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0808_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Gris Ruidoso</button>
          <button id="ep0808_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Mediana 3x3</button>
          <button id="ep0808_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Binarizada (Otsu)</button>
        </div>
      </div>

      <div>
        <button id="ep0808_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generar Formas/Posiciones Aleatorias</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZACIÓN DE LA MATRIZ DE PÍXELES</div>
      <div id="ep0808_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABLA DE MEDICIONES DE LOS OBJETOS (ORDENADOS POR BBOX_X, BBOX_Y)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimeter</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0808_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0808Init) return;
    root.dataset.ep0808Init = "1";

    var H = 10, W = 20, stage = 0;
    var matOrig = [], matMed = [], matBin = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0808_btnRand');
    var elGridContainer = root.querySelector('#ep0808_grid_container');
    var elTbody = root.querySelector('#ep0808_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var w1 = Math.floor(Math.random() * 2) + 4, h1 = Math.floor(Math.random() * 2) + 4;
      var x1 = Math.floor(Math.random() * 2) + 1, y1 = Math.floor(Math.random() * 2) + 1;
      var val1 = 150;
      for (var r = y1; r < y1 + h1; r++) { for (var c = x1; c < x1 + w1; c++) matOrig[r][c] = val1; }

      var w2 = Math.floor(Math.random() * 2) + 4, h2 = Math.floor(Math.random() * 2) + 4;
      var x2 = Math.floor(Math.random() * 2) + 11, y2 = Math.floor(Math.random() * 2) + 2;
      var val2 = 180;
      for (var r = y2; r < y2 + h2; r++) { for (var c = x2; c < x2 + w2; c++) matOrig[r][c] = val2; }

      for (var i = 0; i < 4; i++) {
        var sr = Math.floor(Math.random() * H), sc = Math.floor(Math.random() * W);
        if (matOrig[sr][sc] === 0) matOrig[sr][sc] = 255;
      }
      matOrig[y1 + 1][x1 + 1] = 0; matOrig[y2 + 1][x2 + 1] = 0;

      matMed = JSON.parse(JSON.stringify(matOrig));
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var vals = [];
          for (var dr = -1; dr <= 1; dr++) {
            for (var dc = -1; dc <= 1; dc++) {
              var nr = r + dr, nc = c + dc;
              if (nr >= 0 && nr < H && nc >= 0 && nc < W) { vals.push(matOrig[nr][nc]); } else { vals.push(0); }
            }
          }
          vals.sort(function(a, b){ return a - b; });
          matMed[r][c] = vals[4];
        }
      }

      matBin = matMed.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      medidasObjetos = measureOpenCV(matBin);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matOrig : (stage === 1 ? matMed : matBin);

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          if (stage === 2) {
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else {
            var fgCinza = v > 128 ? '#000000' : '#ffffff';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + v + ',' + v + ',' + v + ');color:' + fgCinza + ';';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após a filtragem.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0808_stage0'), root.querySelector('#ep0808_stage1'), root.querySelector('#ep0808_stage2')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0808_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0808_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0808_stage2').addEventListener('click', function(){ setStage(2, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0808');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.8:** Simulador EP08_08: Filtrado de Mediana en Tonos de Gris y Medición de Objetos OpenCV


In [ ]:
%%writefile EP08_08.py
# Código Python

In [ ]:
TestSuite("EP08_08.py").run()

### EP08_09 🟠 Gradiente de Iluminación y Umbralización Adaptativa

En esta variación, los objetos están inmersos en un fondo con **iluminación no uniforme (gradiente suave de iluminación)**. La umbralización simple por valor único falla, lo que exige un preprocesamiento más robusto.

#### 📋 Directrices de Implementación

1. **Entrada:** imagen en niveles de gris $H \times W$ con variación de fondo de $20$ a $180$.
2. **Preprocesamiento:**
  
* Aplicar **Umbralización Adaptativa** (ej.: `cv2.adaptiveThreshold` con ventana gaussiana de $15 \times 15$ y constante $C = 3$) para aislar los objetos independientemente de la variación del fondo.
  
  `cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, ksize, C) // 255`

  `ksize` y `C` se leen después de la imagen.
  
* Operación morfológica de **Cierre** ($3 \times 3$) para sellar posibles fallos en los contornos.


1. **Medición y Clasificación:** extraer las medidas.


4. **Ordenación y Salida:** ordenar por `(bbox[0], bbox[1])` e imprimir la tabla incluyendo la columna `class`.
   - Para ordenar, usar `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `medidas = mm.measure(img, precision=0.02)`.



#### 📌 Ejemplos

| Entrada | Salida |
|---|---|
| 16 32<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>k 20 | id area perimeter cx cy x y w h circularity solidity vertices<br>1 9.0 12.0 10.0 5.0 8 3 5 5 0.79 1.000 4<br>2 28.3 18.8 25.0 12.0 22 9 7 7 1.00 1.000 3|

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0809" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulador EP08_09: Gradiente de Iluminación y Umbral Adaptativo & Medición OpenCV</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Adaptativo vs Global &rarr; Medición</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:280px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ETAPA DEL PROCESAMIENTO</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0809_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Gradiente de Grises</button>
          <button id="ep0809_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Umbral Global Fallido</button>
          <button id="ep0809_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Umbral Adaptativo Correcto</button>
        </div>
      </div>

      <div>
        <button id="ep0809_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generar Formas/Posiciones Aleatorias</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZACIÓN DE LA MATRIZ DE PÍXELES</div>
      <div id="ep0809_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABLA DE MEDICIONES DE LOS OBJETOS (CALCULADA EN EL UMBRAL ADAPTATIVO CORRECTO)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perímetro</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularidad</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidez</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vértices</th>
            </tr>
          </thead>
          <tbody id="ep0809_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0809Init) return;
    root.dataset.ep0809Init = "1";

    var H = 10, W = 20, stage = 0;
    var matGrad = [], matGlob = [], matAdapt = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0809_btnRand');
    var elGridContainer = root.querySelector('#ep0809_grid_container');
    var elTbody = root.querySelector('#ep0809_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matGrad = Array.from({length: H}, function(_, r){
        return Array.from({length: W}, function(_, c){ return Math.round(20 + c * 9.5); });
      });
      var w1 = 3, h1 = 3;
      var x1 = Math.floor(Math.random() * 2) + 2, y1 = Math.floor(Math.random() * 2) + 2;
      for (var r = y1; r < y1 + h1; r++) { for (var c = x1; c < x1 + w1; c++) matGrad[r][c] += 90; }

      var w2 = 3, h2 = 3;
      var x2 = Math.floor(Math.random() * 2) + 14, y2 = Math.floor(Math.random() * 2) + 2;
      for (var r = y2; r < y2 + h2; r++) { for (var c = x2; c < x2 + w2; c++) matGrad[r][c] += 90; }

      matGlob = matGrad.map(function(row) { return row.map(function(v) { return v > 110 ? 1 : 0; }); });

      var blockSize = 5, half = Math.floor(blockSize / 2), C_val = 20;
      matAdapt = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var sum = 0, count = 0;
          for (var dr = -half; dr <= half; dr++) {
            for (var dc = -half; dc <= half; dc++) {
              var nr = r + dr, nc = c + dc;
              if (nr >= 0 && nr < H && nc >= 0 && nc < W) { sum += matGrad[nr][nc]; count++; }
            }
          }
          var mean = sum / count;
          matAdapt[r][c] = matGrad[r][c] > (mean + C_val) ? 1 : 0;
        }
      }
      medidasObjetos = measureOpenCV(matAdapt);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matGrad : (stage === 1 ? matGlob : matAdapt);

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          if (stage > 0) {
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else {
            var fgCinza = v > 120 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o limiar adaptativo.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0809_stage0'), root.querySelector('#ep0809_stage1'), root.querySelector('#ep0809_stage2')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0809_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0809_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0809_stage2').addEventListener('click', function(){ setStage(2, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0809');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.9:** Simulador EP08_09: Gradiente de Iluminação, Limiar Adaptativo y Medición OpenCV


In [ ]:
%%writefile EP08_09.py
# Código Python

In [ ]:
TestSuite("EP08_09.py").run()

### EP08_10 🔴 Contraste Bajo y Separación de Objetos Tópicos (*Watershed* / Distancia)

En este ejercicio, **algunos objetos geométricos están ligeramente en contacto (superpuestos en los bordes)**. La simple extracción de contornos trataría dos objetos como uno solo.

#### 📋 Directrices de Implementación

1. **Entrada:** matriz $H \times W$ en niveles de gris con objetos de intensidad $110\dots140$ sobre fondo $0$, con ruido y pares de objetos tangentes.
2. **Preprocesamiento y Separación:**
* Aplicación de la umbralización.
* Aplicación de la **Transformada de Distancia** (`mm.dist`).
* Obtención de los picos de distancia para que sirvan como marcadores en la **Transformada *Watershed*** (`mm.watershed`), separando físicamente los objetos en contacto en la máscara. **Consejo:** usar `mm.regmax()` para obtener los máximos locales y luego etiquetar con `mm.label0`.
* Después del *watershed*, aplicar nuevamente la umbralización con `mm.threshold(water,0)//255`.

3. **Análisis de Componentes Conectados:** medir cada región aislada después del *Watershed*.
4. **Salida:** imprimir los componentes ordenados por `(bbox[0], bbox[1])` con sus métricas individuales de área, centroide y solidez.
   - Para ordenar, usar `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `medidas = mm.measure(img, precision=0.02)`.

#### 📌 Ejemplos

| Entrada | Salida |
|---|---|
| 16 16<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>(imagen binaria que contiene dos cuadrados) | id area perimeter cx cy x y w h solidity<br>1 16.0 16.0 5.0 5.0 3 3 5 5 1.000<br>2 16.0 16.0 11.0 5.0 9 3 5 5 1.000 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0810" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulador EP08_10: Separación de Discos Tangentes (Transformada L2 & Watershed)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">mm.dist L2 &rarr; mm.watershed &rarr; Medición</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:300px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ETAPA DEL PROCESAMIENTO MORFOLÓGICO</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0810_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Gris Ruidoso</button>
          <button id="ep0810_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Máscara Unida</button>
          <button id="ep0810_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Distancia L2</button>
          <button id="ep0810_stage3" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">4. Watershed (Corte)</button>
        </div>
      </div>

      <div>
        <button id="ep0810_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generar Discos con Radios Aleatorios</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZACIÓN DE LA MATRIZ DE PÍXELES</div>
      <div id="ep0810_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABLA DE MEDICIONES DE LOS DISCOS DESPUÉS DEL CORTE DEL WATERSHED</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">área</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perímetro</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularidad</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidez</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vértices</th>
            </tr>
          </thead>
          <tbody id="ep0810_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0810Init) return;
    root.dataset.ep0810Init = "1";

    var H = 11, W = 21, stage = 0;
    var matOrig = [], matBin = [], matDist = [], matWash = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0810_btnRand');
    var elGridContainer = root.querySelector('#ep0810_grid_container');
    var elTbody = root.querySelector('#ep0810_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var r1 = Math.floor(Math.random() * 3) + 2, r2 = Math.floor(Math.random() * 3) + 2; 
      var cy1 = Math.floor(Math.random() * 2) + 4, cx1 = Math.floor(Math.random() * 2) + 3;
      var cx2 = cx1 + r1 + r2, cy2 = cy1;

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var d1 = Math.hypot(r - cy1, c - cx1), d2 = Math.hypot(r - cy2, c - cx2);
          if (d1 <= r1 || d2 <= r2) { matOrig[r][c] = 140 + Math.floor(Math.random() * 20); }
        }
      }

      matBin = matOrig.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      matDist = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var fundoPixels = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) { if (matBin[r][c] === 0) fundoPixels.push([r, c]); }
      }

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (matBin[r][c] === 1) {
            var minDist = Infinity;
            for (var k = 0; k < fundoPixels.length; k++) {
              var dist = Math.hypot(r - fundoPixels[k][0], c - fundoPixels[k][1]);
              if (dist < minDist) minDist = dist;
            }
            matDist[r][c] = Math.round(minDist);
          }
        }
      }

      matWash = JSON.parse(JSON.stringify(matBin));
      var colCorte = cx1 + r1; 
      for (var r = 0; r < H; r++) { if (matWash[r][colCorte] === 1) matWash[r][colCorte] = 0; }

      medidasObjetos = measureOpenCV(matWash);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var cell = document.createElement('div');
          if (stage === 0) {
            var v = matOrig[r][c]; cell.textContent = v;
            var fgCinza = v > 120 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';';
          } else if (stage === 1) {
            var v = matBin[r][c]; cell.textContent = v;
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else if (stage === 2) {
            var v = matDist[r][c]; cell.textContent = v;
            var bgDist = v > 0 ? 'rgb(' + (240 - v * 45) + ',' + (240 - v * 30) + ',255)' : '#ffffff';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgDist + ';color:#26241d;';
          } else {
            var v = matWash[r][c]; cell.textContent = v;
            var bgWash = v === 1 ? '#26241d' : '#ffffff', fgWash = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgWash + ';color:' + fgWash + ';';
          }
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o corte do Watershed.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0810_stage0'), root.querySelector('#ep0810_stage1'), root.querySelector('#ep0810_stage2'), root.querySelector('#ep0810_stage3')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0810_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0810_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0810_stage2').addEventListener('click', function(){ setStage(2, this); });
    root.querySelector('#ep0810_stage3').addEventListener('click', function(){ setStage(3, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0810');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.10:** Simulador EP08_10: Separación de Discos Tangentes mediante la Transformada de Distancia L2 y *Watershed*


\newpage

In [ ]:
%%writefile EP08_10.py
# Código Python

In [ ]:
TestSuite("EP08_10.py").run()

### EP08_11 🔴 Clasificación y Validación de Objetos con Plantilla de *Bounding Box*

En este ejercicio, el objetivo es procesar una imagen en tonos de gris que contiene múltiples objetos geométricos, extraer sus propiedades con `mm.measure` y validar las cajas delimitadoras (*bounding boxes*) detectadas en relación con una plantilla real (*Ground Truth* - GT) proporcionada en la entrada, utilizando la métrica IoU (*Intersection over Union*).


#### 📋 Directrices de Implementación

1. **Lectura de la Imagen:** Leer las dimensiones $H \times W$ y la matriz $H \times W$ de píxeles de la imagen en tonos de gris.
2. ***Pipeline* Morfológico:** Binarizar la imagen mediante el método de Otsu (`mm.threshold`) y renderizar la máscara binarizada resultante utilizando `mm.drawImg`.
3. **Lectura de la Plantilla Real (*Ground Truth*):**
   
* Leer la cantidad $G$ de cajas delimitadoras de la plantilla.
* Si $G > 0$, leer $G$ líneas que contienen 5 valores cada una: `id xmin_norm ymin_norm xmax_norm ymax_norm`.
* **Conversión de Coordenadas:** Las coordenadas de la plantilla están normalizadas en el rango $[0.0, 1.0]$. Para convertir a píxeles en la cuadrícula de la imagen:

$$x_{\min} = \lfloor \text{xmin\_norm} \times W \rfloor, \quad y_{\min} = \lfloor \text{ymin\_norm} \times H \rfloor$$


$$w = \lfloor \text{xmax\_norm} \times W \rfloor - x_{\min}, \quad h = \lfloor \text{ymax\_norm} \times H \rfloor - y_{\min}$$


4. **Extracción de Métricas y Cálculo de IoU:**
* Extraer las propiedades de las instancias con `mm.measure(img_bin, precision=0.02)`.
* Para cada *bounding box* detectada $(x, y, w, h)$, calcular la superposición IoU respecto a las cajas de la plantilla y definir `hits = 1` si existe alguna coincidencia con $\text{IoU} \ge 0.50$, o `hits = 0` en caso contrario.


5. **Salida:** Ordenar las instancias por posición `(bbox[0], bbox[1])` e imprimir la tabla CSV con la columna adicional `hits`.
   - Para ordenar, usar `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `medidas = mm.measure(img)`.



#### 🧠 Fundamentación Teórica y Conversión

| Concepto | Fórmula / Operación | Descripción |
| --- | --- | --- |
| **BBox Detectada** | $(x, y, w, h)$ vía `mm.measure` | Caja delimitadora calculada en la cuadrícula discreta en píxeles enteros. |
| **BBox Plantilla (GT)** | $(x_{\min}, y_{\min}, w, h)$ convertidos | Caja real proporcionada en la entrada en coordenadas relativas $[0.0, 1.0]$. |
| **IoU (Intersection over Union)** | $\text{IoU} = \frac{\text{Área}(B_{\text{DET}} \cap B_{\text{GT}})}{\text{Área}(B_{\text{DET}} \cup B_{\text{GT}})}$ | Evalúa la tasa de superposición de las cajas. Se considera válida si $\text{IoU} \ge 0.50$. |
| **Estado de Validación (`hits`)** | $1$ si $\max(\text{IoU}) \ge 0.50$, sino $0$ | Indicador binario de acierto del detector respecto a la plantilla. |



#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* **Línea 1:** Enteros $H$ y $W$ (dimensiones de la matriz).
* **Siguientes $H$ líneas:** $W$ enteros ($0$ a $255$) que representan la imagen en tonos de gris.
* **Línea $H + 2$:** Entero $G$ (cantidad de cajas de la plantilla verdadera).
* **Siguientes $G$ líneas:** 5 valores numéricos por línea: `id xmin_norm ymin_norm xmax_norm ymax_norm` (donde las coordenadas son valores flotantes entre $0.0$ y $1.0$).

**Salida:**

1. Matriz binarizada renderizada vía `mm.drawImg(img_bin)`.
2. Encabezado CSV: `id,area,perimeter,cx,cy,x,y,w,h,circularity,solidity,vertices,hits`
3. Una línea CSV por objeto detectado que contiene sus propiedades formateadas y el indicador `hits` ($1$ o $0$).


#### 📌 Ejemplos

| Entrada | Salida |
|---|---|
| 10 20<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 180 0 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 180 180 180 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 0 180 0 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>2<br>1 0.10 0.30 0.25 0.60<br>2 0.60 0.30 0.75 0.60 | 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 1 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>id area perimeter cx cy x y w h circularity solidity vertices hits<br>1 2.0 5.7 3.0 4.0 2 3 3 3 0.79 1.000 4 1<br>2 4.0 8.0 13.0 4.0 12 3 3 3 0.79 1.000 4 1 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0811" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulador EP08_11: Bounding Boxes y Comparación de IoU con Controles Independientes</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Validación BBox GT vs DET</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:1.5;min-width:220px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MODO DE VISUALIZACIÓN</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0811_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Gris Original</button>
          <button id="ep0811_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Binarizada + Superposiciones</button>
        </div>
      </div>

      <div style="flex:1.5;min-width:240px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">VISUALIZACIÓN DE LAS BOUNDING BOXES</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0811_toggleGT" style="padding:5px 10px;font-size:10.5px;font-weight:700;border:1px solid #1d4ed8;background:#2563eb;color:#ffffff;cursor:pointer;border-radius:8px;white-space:nowrap;display:inline-flex;align-items:center;gap:5px;"><span>🟦</span> BBox Plantilla (GT)</button>
          <button id="ep0811_toggleDET" style="padding:5px 10px;font-size:10.5px;font-weight:700;border:1px solid #047857;background:#059669;color:#ffffff;cursor:pointer;border-radius:8px;white-space:nowrap;display:inline-flex;align-items:center;gap:5px;"><span>🟩</span> BBox Detectada (DET)</button>
        </div>
      </div>

      <div>
        <button id="ep0811_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Generar Escenas Aleatorias</button>
      </div>
    </div>

    <div style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:10px;padding:8px 12px;margin-bottom:12px;display:flex;gap:16px;flex-wrap:wrap;align-items:center;justify-content:center;">
      <span style="font-size:9.5px;font-weight:700;color:#8a8371;margin-right:4px;">LEYENDA DE LAS BBOXES:</span>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#2563eb;border:1px dashed #93c5fd;"></span> <span>Plantilla Real (GT)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#059669;border:1px solid #34d399;"></span> <span>Detección Aceptada (IoU &ge; 0.5)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#dc2626;border:1px solid #f87171;"></span> <span>Detección Rechazada (IoU &lt; 0.5)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#7c3aed;border:1px double #a78bfa;"></span> <span>Superposición de BBoxes</span></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZACIÓN DE LA MATRIZ DE PÍXELES</div>
      <div id="ep0811_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MEDIDAS, CLASIFICACIÓN GEOMÉTRICA Y COMPARACIÓN IOU CON PLANTILLA</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">clase</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidez</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vértices</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox det (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox gt (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">IoU</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">estado (IoU &ge; 0.5)</th>
            </tr>
          </thead>
          <tbody id="ep0811_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0811Init) return;
    root.dataset.ep0811Init = "1";

    var H = 10, W = 20, stage = 0;
    var showGT = true, showDET = true;
    // matOrig/matBin: cena. medidasObjetos: 1 registro por objeto real (GT), casado com sua melhor DET.
    // detBoxesAtuais: caixas "detectadas" simuladas (com ruído/deslocamento em relação ao objeto real).
    var matOrig = [], matBin = [], medidasObjetos = [], detBoxesAtuais = [];

    var elBtnRand = root.querySelector('#ep0811_btnRand');
    var elToggleGT = root.querySelector('#ep0811_toggleGT');
    var elToggleDET = root.querySelector('#ep0811_toggleDET');
    var elGridContainer = root.querySelector('#ep0811_grid_container');
    var elTbody = root.querySelector('#ep0811_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function calculateIoU(boxA, boxB) {
      var ax1 = boxA.x, ay1 = boxA.y, ax2 = boxA.x + boxA.w, ay2 = boxA.y + boxA.h;
      var bx1 = boxB.x, by1 = boxB.y, bx2 = boxB.x + boxB.w, by2 = boxB.y + boxB.h;
      var ix1 = Math.max(ax1, bx1), iy1 = Math.max(ay1, by1);
      var ix2 = Math.min(ax2, bx2), iy2 = Math.min(ay2, by2);
      var iw = Math.max(0, ix2 - ix1), ih = Math.max(0, iy2 - iy1);
      var inter = iw * ih;
      var areaA = boxA.w * boxA.h, areaB = boxB.w * boxB.h;
      var union = areaA + areaB - inter;
      return union > 0 ? inter / union : 0.0;
    }

    // FIX: bbox extraída dos pixels reais do objeto (mat) é o GABARITO (GT) — é a posição
    // verdadeira e exata do objeto na cena. As caixas em `detBoxes` (com deslocamento
    // aleatório) representam a saída ruidosa de um detector, e são casadas ao GT mais
    // próximo por IoU.
    function measureOpenCV(mat, detBoxes) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var gtBbox = cv2_boundingRect(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        var solidity = hull_area > 0 ? area / hull_area : 0;
        var classe = solidity < 0.85 ? "Cruz (Côncavo)" : "Retângulo (Convexo)";
        var bestIoU = 0.0, matchedDET = { x: 0, y: 0, w: 0, h: 0 };

        detBoxes.forEach(function(d) {
          var iou = calculateIoU(gtBbox, d);
          if (iou > bestIoU) { bestIoU = iou; matchedDET = d; }
        });

        medidas.push({
          classe: classe, area: area, gtBox: gtBbox, detBox: matchedDET,
          solidity: solidity, vertices: poly.length, iou: bestIoU, ok: bestIoU >= 0.50
        });
      });
      medidas.sort(function(a, b) { return a.gtBox.x !== b.gtBox.x ? a.gtBox.x - b.gtBox.x : a.gtBox.y - b.gtBox.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var armLen = Math.floor(Math.random() * 2) + 1, thick = 1;
      var w1 = armLen * 2 + thick, h1 = armLen * 2 + thick;
      var x1 = Math.floor(Math.random() * Math.max(1, 8 - w1)) + 1;
      var y1 = Math.floor(Math.random() * Math.max(1, H - h1)) + 1;

      for (var r = 0; r < h1; r++) {
        for (var c = 0; c < w1; c++) {
          if ((c >= armLen && c < armLen + thick) || (r >= armLen && r < armLen + thick)) { matOrig[y1 + r][x1 + c] = 180; }
        }
      }

      var w2 = Math.floor(Math.random() * 3) + 3, h2 = Math.floor(Math.random() * 3) + 3;
      var x2 = Math.floor(Math.random() * Math.max(1, W - 10 - w2)) + 10;
      var y2 = Math.floor(Math.random() * Math.max(1, H - h2)) + 1;

      for (var r = y2; r < y2 + h2; r++) {
        for (var c = x2; c < x2 + w2; c++) { matOrig[r][c] = 180; }
      }

      // Estas caixas simulam a saída de um DETECTOR real: deslocadas/imprecisas em relação
      // ao objeto verdadeiro (que será obtido depois via segmentação em matBin -> GT).
      var shiftX1 = Math.random() > 0.5 ? 1 : 0, shiftY1 = Math.random() > 0.5 ? 1 : 0;
      var shiftX2 = Math.random() > 0.6 ? -2 : 0;

      detBoxesAtuais = [
        { x: Math.max(0, x1 + shiftX1), y: Math.max(0, y1 + shiftY1), w: w1, h: h1 },
        { x: Math.max(0, x2 + shiftX2), y: y2, w: w2 + (shiftX2 !== 0 ? 2 : 0), h: h2 }
      ];

      matBin = matOrig.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      medidasObjetos = measureOpenCV(matBin, detBoxesAtuais);
      renderGrid();
      renderTabela();
    }

    function inBox(r, c, box) { return r >= box.y && r < box.y + box.h && c >= box.x && c < box.x + box.w; }
    function isBoxEdge(r, c, box) { if (!inBox(r, c, box)) return false; return r === box.y || r === box.y + box.h - 1 || c === box.x || c === box.x + box.w - 1; }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matOrig : matBin;

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          var isGT = false, isDetOK = false, isDetFail = false;

          if (stage === 1) {
            // FIX: GT agora vem de m.gtBox (bbox real extraída dos pixels do objeto)
            if (showGT) { medidasObjetos.forEach(function(m) { if (isBoxEdge(r, c, m.gtBox)) isGT = true; }); }
            // FIX: DET agora vem de m.detBox (bbox ruidosa casada por IoU)
            if (showDET) {
              medidasObjetos.forEach(function(m) {
                if (isBoxEdge(r, c, m.detBox)) { if (m.ok) isDetOK = true; else isDetFail = true; }
              });
            }

            if (isGT && isDetOK) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#7c3aed;color:#ffffff;border:2px double #a78bfa;box-sizing:border-box;';
            } else if (isGT && isDetFail) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#c026d3;color:#ffffff;border:2px double #f472b6;box-sizing:border-box;';
            } else if (isDetOK) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#059669;color:#ffffff;border:2px solid #34d399;box-sizing:border-box;';
            } else if (isDetFail) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#dc2626;color:#ffffff;border:2px solid #f87171;box-sizing:border-box;';
            } else if (isGT) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#2563eb;color:#ffffff;border:2px dashed #93c5fd;box-sizing:border-box;';
            } else {
              var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';border:none;';
            }
          } else {
            var fgCinza = v > 100 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';border:none;';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="9" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado na cena.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        // FIX: bboxDet vem de m.detBox (caixa ruidosa) e bboxGT vem de m.gtBox (caixa real)
        var bboxDet = '(' + m.detBox.x + ',' + m.detBox.y + ',' + m.detBox.w + ',' + m.detBox.h + ')';
        var bboxGT = '(' + m.gtBox.x + ',' + m.gtBox.y + ',' + m.gtBox.w + ',' + m.gtBox.h + ')';
        var statusHtml = m.ok ? '<span style="color:#27ae60;font-weight:bold;">✔ True</span>' : '<span style="color:#e74c3c;font-weight:bold;">✖ False</span>';

        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.classe + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxDet + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxGT + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.iou.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + statusHtml + '</td>';
        elTbody.appendChild(tr);
      });
    }

    elToggleGT.addEventListener('click', function(){
      showGT = !showGT;
      if (showGT) {
        this.style.background = '#2563eb'; this.style.borderColor = '#1d4ed8'; this.style.color = '#ffffff';
        this.querySelector('span').textContent = '🟦';
      } else {
        this.style.background = '#f1ead7'; this.style.borderColor = '#d4cebe'; this.style.color = '#5e5a4a';
        this.querySelector('span').textContent = '⬜';
      }
      renderGrid();
    });

    elToggleDET.addEventListener('click', function(){
      showDET = !showDET;
      if (showDET) {
        this.style.background = '#059669'; this.style.borderColor = '#047857'; this.style.color = '#ffffff';
        this.querySelector('span').textContent = '🟩';
      } else {
        this.style.background = '#f1ead7'; this.style.borderColor = '#d4cebe'; this.style.color = '#5e5a4a';
        this.querySelector('span').textContent = '⬜';
      }
      renderGrid();
    });

    elBtnRand.addEventListener('click', gerarCenario);

    function setStage(s, btn){
      [root.querySelector('#ep0811_stage0'), root.querySelector('#ep0811_stage1')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    root.querySelector('#ep0811_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0811_stage1').addEventListener('click', function(){ setStage(1, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0811');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.11:** Simulador EP08_11: Clasificación Geométrica con Controles Independientes de *Overlays BBox* (GT y DET)


In [ ]:
%%writefile EP08_11.py
# Código Python

In [ ]:
TestSuite("EP08_11.py").run()

### EP08_12 🔴 Segmentación de Instancias en Imagen Real: Objetos Geométricos

El ejemplo de segmentación clásica de este capítulo separó "instancias" de monedas por desconexión espacial en la máscara binaria resultante de la umbralización de Otsu. En este ejercicio vas a aplicar la misma idea — pero ahora sobre una imagen real con objetos geométricos variados — encadenando preprocesamiento, binarización, extracción de contornos (`cv2.findContours`) y validación del resultado contra un patrón de *bounding boxes*.

A diferencia del ejercicio anterior (etiquetado sobre máscara ya preparada), aquí partes de la **imagen original**: la calidad de tu segmentación depende directamente de las elecciones de preprocesamiento (filtrado, umbralización, operaciones morfológicas) realizadas antes de etiquetar los componentes.

#### 📋 Directrices de Implementación

1. **Entrada:** utilizar la imagen `00000.jpg`.
2. **Preprocesamiento y segmentación:** aplicar las etapas necesarias (filtrado, binarización y operaciones morfológicas) para separar automáticamente los objetos del fondo, sin recortes manuales.
3. **Etiquetado y medición:** para cada objeto segmentado, determinar:
   - área;
   - centro de masa (centroide);
   - tipo, conforme al conjunto `obj2`.
4. **Anotación visual:** escribir, en el interior de cada objeto, su área y la sigla del tipo (`obj2`).
5. **Validación (IoU):** calcular el *Intersection over Union* (IoU) entre el *bounding box* detectado (`cv2.boundingRect`) y el *bounding box* de patrón del tipo correspondiente. Un objeto se considera correctamente segmentado solo si hay exactamente un *bounding box* del tipo correcto con **IoU ≥ 0,5**.
6. **Salida:** imprimir, para cada objeto detectado, su identificador, tipo y si fue validado con éxito (`acertou=1`) o no. La impresión debe seguir el orden de las clases de `obj2` (0=Tria … 8=Cruz); dentro de la misma clase, ordenar los objetos por la coordenada vertical del centroide (`cy`) creciente. Al final, imprimir la precisión general.

#### 📌 Restricciones Computacionales

* **Sin recorte manual:** toda la segmentación debe realizarse sobre la imagen completa.
* **Conjunto de clases fijo:**
  ```python
  obj  = ['Triangulo','Quadrado','Pendagono','Hexagono','Heptagono','Circulo',
          'Elipse','Estrela','Cruz']
  obj2 = ['Tria','Quad','Pent','Hexa','Hept','Circ','Elip','Estr','Cruz']
  ```
* **Dimensión de la imagen:** 608×608 píxeles — usada para desnormalizar las coordenadas del archivo TXT.
* **Validación por centro de masa:** un objeto solo se considera correctamente segmentado si su centroide está estrictamente dentro del *boundbox* del patrón correspondiente al mismo tipo de objeto.

#### 🧠 Fundamentación Teórica

| Elemento | Papel en la segmentación de instancias |
|---|---|
| Preprocesamiento (filtrado, umbralización) | Etapa que produce la máscara binaria a partir de la imagen de intensidad original |
| `cv2.findContours` | Extrae los contornos de los componentes conectados en la máscara binaria |
| Momentos geométricos (`cv2.moments`) | Permiten calcular el centro de masa (centroide) de cada contorno |
| `approxPolyDP` / vértices | Ayuda en la clasificación del tipo de objeto (nº de lados aproximado) |
| Validación mediante *boundbox* | Confirma si la instancia segmentada corresponde espacialmente a un objeto del patrón, midiendo la precisión del método |

#### 📌 Ejemplo de Salida Esperada

```
Objeto 1: tipo=Tria, validado=True
...
Precisión: 88.89%
```

**Parámetros fijos para reproducibilidad:** para que la salida coincida con el patrón de corrección automática, utiliza exactamente: filtro de área mínima de 300 píxeles; `cv2.approxPolyDP` con `epsilon = 0.02 * perímetro`; umbral de solidez 0.92 y conteo de vértices ≥ 9 (con ≥ 11 para diferenciar Cruz de Estrella) para formas cóncavas; relación de aspecto 1.15 para diferenciar Círculo de Elipse; umbral de IoU = 0.5 en la validación.

#### 📌 Archivos de Referencia (`.jpg` y `.txt`)

Para depuración local, se proporcionan dos archivos de referencia (incluidos en esta entrega; al integrarlos al repositorio del capítulo, guárdalos en `all/cap08/dados/EP08/`):

* 📥 **Imagen (`00000.jpg`)**: imagen de objetos geométricos utilizada como entrada del ejercicio. El objetivo es segmentar automáticamente cada objeto, determinar su tipo y calcular sus medidas.
* 📥 **Patrón (`00000.txt`)**: archivo que contiene las *bounding boxes* normalizadas de los objetos presentes en la imagen. Cada línea posee el identificador de la clase y las coordenadas normalizadas de las esquinas superior izquierda e inferior derecha, utilizándose para validar automáticamente la segmentación.

La [Figura 8.12](#fig-08-ep12) presenta la imagen de entrada y la misma imagen con las *bounding boxes* dibujadas a partir del archivo de patrón.

In [ ]:
import os
import urllib.request
from morph import mm

def garantir_e_baixar(nome):
    pasta = "dados/EP12"
    caminho = os.path.join(pasta, nome)

    os.makedirs(pasta, exist_ok=True)

    if not os.path.exists(caminho):
        url = (
            "https://raw.githubusercontent.com/"
            "fzampirolli/pdi-vc/master/all/cap08/dados/EP12/"
            + nome
        )
        print(f"Baixando {nome}...")
        urllib.request.urlretrieve(url, caminho)

    return caminho

img_arq = garantir_e_baixar("00000.jpg")
txt_arq = garantir_e_baixar("00000.txt")

img = mm.read(img_arq)
img_bb = mm.showBoundBox(img, txt_arq, fmt="yolo", show=False)

mm.show(
    [img, img_bb],
    titles=[
        "Imagem original",
        "Bounding boxes del gabarito"
    ],
    cols=2,
    figsize=(10,5)
)

**Figura 8.12:** Simulador EP08_12: Imagem utilizada no EP08_12. À esquerda, a imagem original. À direita, a imagem com as *bounding boxes* do arquivo de gabarito.


In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0812" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulador EP08_12: Precisión de Segmentación en Múltiples Objetos</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">🟢 acertó si IoU ≥ umbral Y tipo correcto</span>
  </div>


  <div style="padding:20px;background:white;overflow:auto">

    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Cada forma tiene un <i>boundbox</i> de referencia (rectángulo discontinuo, justo alrededor de la forma) y un <i>boundbox</i> detectado (rectángulo sólido, desplazado/ruidoso). Ajuste el ruido, el sesgo y el umbral de IoU para ver cómo cambia la validación.
    </p>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#c0392b;">ruido de segmentación (px, jitter máx. por lado)</label><span id="ep0812_ruido_v" style="font-family:monospace;font-weight:bold;color:#c0392b;">0</span></div>
        <input id="ep0812_ruido" style="width:100%;accent-color:#c0392b;" max="20" min="0" step="1" type="range" value="0">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">sesgo sistemático en x (px)</label><span id="ep0812_bias_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">0</span></div>
        <input id="ep0812_bias" style="width:100%;accent-color:#2980b9;" max="20" min="-20" step="1" type="range" value="0">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#27ae60;">umbral de IoU</label><span id="ep0812_thr_v" style="font-family:monospace;font-weight:bold;color:#27ae60;">0.50</span></div>
        <input id="ep0812_thr" style="width:100%;accent-color:#27ae60;" max="0.9" min="0.1" step="0.05" type="range" value="0.5">
      </div>
      <div style="display:flex;align-items:center;gap:8px;">
        <input id="ep0812_erro" type="checkbox" style="accent-color:#8e44ad;width:16px;height:16px;">
        <label style="font-size:12px;font-weight:bold;color:#8e44ad;">simular error de clasificación (2 objetos con tipo intercambiado)</label>
      </div>
    </div>

<div id="ep0812_svg" style="width:100%;max-width:420px;margin:0 auto 16px auto;"></div>

    <table style="width:100%;border-collapse:collapse;font-size:11px;font-family:monospace;margin-bottom:12px;">
      <thead>
        <tr style="background:#f3efe6;">
          <th style="padding:4px;border:1px solid #ddd;">id</th>
          <th style="padding:4px;border:1px solid #ddd;">tipo real</th>
          <th style="padding:4px;border:1px solid #ddd;">tipo detectado</th>
          <th style="padding:4px;border:1px solid #ddd;">IoU</th>
          <th style="padding:4px;border:1px solid #ddd;">≥ umbral</th>
          <th style="padding:4px;border:1px solid #ddd;">acertó</th>
        </tr>
      </thead>
      <tbody id="ep0812_tbody"></tbody>
    </table>

    <div id="ep0812_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:12px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>

function svgNS(tag){
  var SVG_NS = "http" + "://www.w3.org/2000/svg";
  return document.createElementNS(SVG_NS, tag);
}

(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var ruidoEl = root.querySelector('#ep0812_ruido'), ruidovEl = root.querySelector('#ep0812_ruido_v');
    var biasEl = root.querySelector('#ep0812_bias'), biasvEl = root.querySelector('#ep0812_bias_v');
    var thrEl = root.querySelector('#ep0812_thr'), thrvEl = root.querySelector('#ep0812_thr_v');
    var erroEl = root.querySelector('#ep0812_erro');
    var svgWrap = root.querySelector('#ep0812_svg');
    var svgEl = svgNS('svg');
    svgEl.setAttribute('viewBox', '0 0 360 340');
    svgEl.setAttribute('style', 'width:100%;display:block;background:#0d0d0d;border-radius:12px;border:1px solid #333;');
    svgWrap.appendChild(svgEl);

    var tbody = root.querySelector('#ep0812_tbody');
    var dbg = root.querySelector('#ep0812_debug');

    var TIPOS = ['Circ','Tria','Quad','Cruz','Pent','Hexa','Hept','Estr','Elip'];
    var CORES = ['#a33a5b','#8fe3b0','#c98a7a','#5c4a5e','#3fbf5f','#d9c832','#e08a2b','#5c5470','#4a5a3a'];
    var FATOR = [1.0, 1.3, 0.7, 1.5, 0.9, 1.1, 0.6, 1.4, 0.8]; // sensibilidade individual ao ruído (fixa)
    var ANGS  = [30, 160, 260, 5, 200, 90, 340, 130, 240];      // direção fixa do erro por objeto (graus)
    var SCALE = [0.9, 1.15, 0.8, 1.2, 1.0, 0.95, 1.1, 1.25, 0.85]; // fator de encolhimento/expansão do bbox (fixo)

    var OBJS = [
      {cx:55,  cy:45,  r:22},
      {cx:150, cy:70,  r:24},
      {cx:250, cy:50,  r:22},
      {cx:320, cy:130, r:20},
      {cx:90,  cy:190, r:24},
      {cx:190, cy:220, r:24},
      {cx:275, cy:190, r:24},
      {cx:315, cy:270, r:22},
      {cx:150, cy:150, r:18}
    ];

    function svgShape(tipo, cx, cy, r, cor){
      var s = '';
      if(tipo==='Circ'){
        s = '<circle cx="'+cx+'" cy="'+cy+'" r="'+r+'" fill="'+cor+'"/>';
      } else if(tipo==='Elip'){
        s = '<ellipse cx="'+cx+'" cy="'+cy+'" rx="'+(r*1.1)+'" ry="'+(r*0.65)+'" fill="'+cor+'"/>';
      } else if(tipo==='Quad'){
        s = '<rect x="'+(cx-r*0.8)+'" y="'+(cy-r*0.8)+'" width="'+(r*1.6)+'" height="'+(r*1.6)+'" fill="'+cor+'"/>';
      } else if(tipo==='Cruz'){
        var w = r*0.5, l = r*1.4;
        s = '<g fill="'+cor+'">'+
            '<rect x="'+(cx-w/2)+'" y="'+(cy-l/2)+'" width="'+w+'" height="'+l+'"/>'+
            '<rect x="'+(cx-l/2)+'" y="'+(cy-w/2)+'" width="'+l+'" height="'+w+'"/></g>';
      } else {
        var sides = {Tria:3, Pent:5, Hexa:6, Hept:7}[tipo];
        var isStar = (tipo==='Estr');
        var pts = [];
        if(isStar){
          var spikes=5, outer=r, inner=r*0.45;
          for(var i=0;i<spikes*2;i++){
            var rad = (i%2===0)?outer:inner;
            var ang = Math.PI/spikes*i - Math.PI/2;
            pts.push((cx+rad*Math.cos(ang)).toFixed(1)+','+(cy+rad*Math.sin(ang)).toFixed(1));
          }
        } else {
          for(var i=0;i<sides;i++){
            var ang = 2*Math.PI/sides*i - Math.PI/2;
            pts.push((cx+r*Math.cos(ang)).toFixed(1)+','+(cy+r*Math.sin(ang)).toFixed(1));
          }
        }
        s = '<polygon points="'+pts.join(' ')+'" fill="'+cor+'"/>';
      }
      return s;
    }

    // bbox "verdadeiro": retângulo justo em torno da forma (sem folga artificial)
    function trueBBox(tipo, cx, cy, r){
      var hw = r, hh = r;
      if(tipo==='Elip'){ hw = r*1.1; hh = r*0.65; }
      else if(tipo==='Quad'){ hw = r*0.8; hh = r*0.8; }
      else if(tipo==='Cruz'){ hw = r*0.7; hh = r*0.7; }
      return [cx-hw, cy-hh, cx+hw, cy+hh];
    }

    function iou(a, b){
      var x1 = Math.max(a[0], b[0]), y1 = Math.max(a[1], b[1]);
      var x2 = Math.min(a[2], b[2]), y2 = Math.min(a[3], b[3]);
      var inter = Math.max(0, x2-x1) * Math.max(0, y2-y1);
      var areaA = (a[2]-a[0])*(a[3]-a[1]);
      var areaB = (b[2]-b[0])*(b[3]-b[1]);
      var uni = areaA + areaB - inter;
      return uni > 0 ? inter/uni : 0;
    }

    function render(){
      var ruido = parseInt(ruidoEl.value);
      var bias = parseInt(biasEl.value);
      var thr = parseFloat(thrEl.value);
      var erroAtivo = erroEl.checked;
      ruidovEl.textContent = ruido;
      biasvEl.textContent = bias;
      thrvEl.textContent = thr.toFixed(2);

      var svgContent = '';
      var rows = '';
      var acertos = 0;

      OBJS.forEach(function(o, i){
        var tipoReal = TIPOS[i];
        var cor = CORES[i];

        var gtBox = trueBBox(tipoReal, o.cx, o.cy, o.r);
        svgContent += '<g opacity="0.9">'+svgShape(tipoReal, o.cx, o.cy, o.r, cor)+'</g>';
        svgContent += '<rect x="'+gtBox[0]+'" y="'+gtBox[1]+'" width="'+(gtBox[2]-gtBox[0])+'" height="'+(gtBox[3]-gtBox[1])+'" fill="none" stroke="#aaa" stroke-dasharray="4,3" stroke-width="1.2"/>';

        // bbox detectado: escala fixa individual + jitter (ruído) + viés em x
        var scl = SCALE[i];
        var mag = ruido * FATOR[i];
        var ang = ANGS[i] * Math.PI/180;
        var jx = mag*Math.cos(ang), jy = mag*Math.sin(ang);
        var dw = (gtBox[2]-gtBox[0]) * scl, dh = (gtBox[3]-gtBox[1]) * scl;
        var dcx = o.cx + jx + bias, dcy = o.cy + jy;
        var detBox = [dcx-dw/2, dcy-dh/2, dcx+dw/2, dcy+dh/2];

        var val = iou(gtBox, detBox);
        var passaLimiar = val >= thr;

        var corDet = passaLimiar ? '#27ae60' : '#c0392b';
        svgContent += '<rect x="'+detBox[0]+'" y="'+detBox[1]+'" width="'+(detBox[2]-detBox[0])+'" height="'+(detBox[3]-detBox[1])+'" fill="none" stroke="'+corDet+'" stroke-width="1.6"/>';

        var tipoDetectado = tipoReal;
        if(erroAtivo && (i===1 || i===6)){
          tipoDetectado = TIPOS[(i+2)%TIPOS.length];
        }
        var tipoCorreto = (tipoDetectado === tipoReal);
        var acertou = passaLimiar && tipoCorreto;
        if(acertou) acertos++;

        svgContent += '<text x="'+(o.cx)+'" y="'+(gtBox[1]-6)+'" font-size="9" fill="#ccc" text-anchor="middle" font-family="monospace">'+ (i+1) +'</text>';

        rows += '<tr>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+(i+1)+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+tipoReal+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;'+(tipoCorreto?'':'color:#c0392b;font-weight:bold;')+'">'+tipoDetectado+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+val.toFixed(2)+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+(passaLimiar?'sim':'não')+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;'+(acertou?'color:#27ae60;font-weight:bold;':'color:#c0392b;font-weight:bold;')+'">'+(acertou?'✔':'✘')+'</td>'+
          '</tr>';
      });

      svgEl.innerHTML = svgContent;
      tbody.innerHTML = rows;

      var acc = (acertos/OBJS.length*100).toFixed(1);
      dbg.textContent = 'Objetos validados: '+acertos+' / '+OBJS.length+'  →  Acurácia = '+acc+'%';
    }

    ruidoEl.addEventListener('input', render);
    biasEl.addEventListener('input', render);
    thrEl.addEventListener('input', render);
    erroEl.addEventListener('change', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0812');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.13:** Simulador EP08_12: Precisión de Segmentación en Múltiples Objetos (IoU)


In [ ]:
%%writefile EP08_12.py
# Código Python

In [ ]:
TestSuite("EP08_12.py").run()